<a href="https://colab.research.google.com/github/txebas/5-Day-Gen-AI-Intensive-Course-with-Google/blob/main/LIGHTPIPELINERAGV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install rank_bm25

In [12]:
pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 20.0 MB/s eta 0:00:00


In [ ]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 76.8 MB/s eta 0:00:00


In [ ]:
import os
import time
import json
import itertools
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass, asdict
from tabulate import tabulate
from sentence_transformers import SentenceTransformer
import faiss

# Configuración simulada para ejecución local (Reemplazar con rutas o modelos reales)
# Nota: Para los SLMs usamos la interfaz de HuggingFace optimizada, pero se puede acoplar a llama.cpp
# indicando simplemente el cambio en la clase correspondiente.

# ==========================================
# 0. ESTRUCTURAS DE DATOS (DENTRO DEL PIPELINE)
# ==========================================

@dataclass
class Chunk:
    id: str
    text: str
    metadata: Dict[str, Any]

@dataclass
class ExperimentoResultado:
    config_id: str
    chunk_strategy: str
    embedding_model: str
    retrieval_strategy: str
    reranker_model: str
    slm_model: str
    tiempo_ejecucion_seg: float
    respuesta: str
    tokens_recuperados: int

# ==========================================
# 1. FASE DE CHUNKING (ESTRATEGIAS)
# ==========================================

class ChunkingStrategy:
    def execute(self, text: str) -> List[Chunk]:
        raise NotImplementedError

class RecursiveCharacterChunking(ChunkingStrategy):
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def execute(self, text: str) -> List[Chunk]:
        # Simulación de split recursivo ingenuo
        # En producción usar: from langchain.text_splitter import RecursiveCharacterTextSplitter
        chunks = []
        words = text.split()
        step = self.chunk_size // 5 # Estimación grosera de palabras
        overlap_step = self.chunk_overlap // 5

        idx = 0
        chunk_id = 0
        while idx < len(words):
            chunk_words = words[idx:idx+step]
            chunk_text = " ".join(chunk_words)
            chunks.append(Chunk(id=f"rec_{chunk_id}", text=chunk_text, metadata={"type": "recursive"}))
            idx += (step - overlap_step)
            chunk_id += 1
        return chunks

class StructureAwareChunking(ChunkingStrategy):
    def execute(self, text: str) -> List[Chunk]:
        # Fragmentación basada en encabezados Markdown (#, ##, ###)
        # Ideal para separar secciones macro de un 10-K sin mezclar tablas fiscales
        chunks = []
        sections = text.split("\n## ")
        for idx, section in enumerate(sections):
            if not section.strip():
                continue
            chunks.append(Chunk(id=f"struct_{idx}", text=f"## {section}", metadata={"type": "structure_aware"}))
        return chunks

class ParentChildChunking(ChunkingStrategy):
    def execute(self, text: str) -> List[Chunk]:
        # Divide en bloques grandes (Padre) y genera sub-fragmentos indexables (Hijos)
        # Devuelve los hijos con referencia cruzada en metadata al padre
        parent_sections = text.split("\n## ")
        chunks = []
        child_id_counter = 0

        for p_idx, p_text in enumerate(parent_sections):
            if not p_text.strip():
                continue
            parent_content = f"## {p_text}"
            # Generar hijos pequeños (ej. cada 2 párrafos)
            paragraphs = parent_content.split("\n\n")
            for c_idx in range(0, len(paragraphs), 2):
                child_text = "\n\n".join(paragraphs[c_idx:c_idx+2])
                if len(child_text.strip()) > 20:
                    chunks.append(Chunk(
                        id=f"child_{child_id_counter}",
                        text=child_text,
                        metadata={"type": "child", "parent_text": parent_content, "parent_id": f"parent_{p_idx}"}
                    ))
                    child_id_counter += 1
        return chunks

# ==========================================
# 2. FASE DE EMBEDDINGS E INDEXACIÓN (FAISS LOCAL)
# ==========================================
import numpy as np
try:
    import faiss
    from sentence_transformers import SentenceTransformer
except ImportError:
    # Fallback mock si corres en un entorno sin dependencias instaladas
    pass

class VectorStoreIndex:
    def __init__(self, model_name: str, chunks: List[Chunk]):
        self.model_name = model_name
        self.chunks = chunks
        # Para evitar sobrecargar la memoria en la permutación masiva, inicializamos bajo demanda
        self.model = SentenceTransformer(model_name)
        self.dimension = self.model.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatIP(self.dimension) # Inner Product para Similitud Coseno con vectores normalizados

        # Construir índice
        texts = [c.text for c in chunks]
        embeddings = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        # Normalizar para similitud coseno exacta
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)

    def search_vector(self, query: str, top_k: int = 50) -> List[Tuple[Chunk, float]]:
        query_emb = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_emb)
        distances, indices = self.index.search(query_emb, top_k)

        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx != -1:
                results.append((self.chunks[idx], float(dist)))
        return results

# ==========================================
# 3. FASE DE RECUPERACIÓN (RETRIEVAL HÍBRIDO & BM25)
# ==========================================
from rank_bm25 import BM25Okapi

class RetrievalEngine:
    def __init__(self, vector_index: VectorStoreIndex, chunks: List[Chunk]):
        self.vector_index = vector_index
        self.chunks = chunks
        # Inicializar BM25 léxico
        tokenized_corpus = [c.text.lower().split(" ") for c in chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def retrieve(self, query: str, strategy: str, top_k: int = 50) -> List[Chunk]:
        if strategy == "R4.1_COSINE":
            res = self.vector_index.search_vector(query, top_k=top_k)
            return [chunk for chunk, _ in res]

        elif strategy == "R4.2_HYBRID_RRF":
            # 1. Recuperación Densa
            dense_res = self.vector_index.search_vector(query, top_k=top_k)
            dense_ids = [c.id for c, _ in dense_res]

            # 2. Recuperación Léxica BM25
            tokenized_query = query.lower().split(" ")
            bm25_scores = self.bm25.get_scores(tokenized_query)
            top_bm25_indices = np.argsort(bm25_scores)[::-1][:top_k]
            bm25_ids = [self.chunks[idx].id for idx in top_bm25_indices]

            # 3. Fusión mediante Reciprocal Rank Fusion (RRF)
            rrf_scores = {}
            k_rrf = 60 # Constante estándar RRF

            for rank, cid in enumerate(dense_ids):
                rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k_rrf + rank + 1))
            for rank, cid in enumerate(bm25_ids):
                rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k_rrf + rank + 1))

            # Ordenar por el score fusionado
            sorted_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

            # Recomponer lista de Chunks ordenados
            chunks_dict = {c.id: c for c in self.chunks}
            final_chunks = []
            for cid, _ in sorted_ids:
                chunk_obj = chunks_dict[cid]
                # Modificación para la Ruta C: Parent-Child
                if chunk_obj.metadata.get("type") == "child":
                    # Si la estrategia requiere resolver el padre, devolvemos un chunk modificado con el contexto macro
                    parent_text = chunk_obj.metadata.get("parent_text", chunk_obj.text)
                    final_chunks.append(Chunk(id=chunk_obj.id, text=parent_text, metadata=chunk_obj.metadata))
                else:
                    final_chunks.append(chunk_obj)
            return final_chunks

        return []

# ==========================================
# 4. FASE DE RE-RANKING (CROSS-ENCODERS)
# ==========================================
class ReRanker:
    def __init__(self, model_name: str):
        self.model_name = model_name
        # Si es baseline "None", no cargamos modelo para ahorrar recursos
        if model_name != "None":
            # Simulación o inicialización de bge-reranker-base / local
            pass

    def rerank(self, query: str, candidates: List[Chunk], top_n: int = 5) -> List[Chunk]:
        if self.model_name == "None" or not candidates:
            return candidates[:top_n]

        # Simulación de ordenamiento Cross-Encoder ligero (bge-reranker-base)
        # En entorno real usar: from sentence_transformers import CrossEncoder
        # scores = model.predict([(query, c.text) for c in candidates])
        # Aquí simulamos basándonos en una ligera perturbación para emular cómputo local veloz
        return candidates[:top_n]

# ==========================================
# 5. FASE DE GENERACIÓN (MOCK / WRAPPER SLM LOCAL)
# ==========================================
class SmallLanguageModel:
    def __init__(self, model_id: str):
        self.model_id = model_id

    def generate(self, query: str, context_chunks: List[Chunk]) -> str:
        # Construcción de un prompt financiero estricto basado en anclaje semántico
        context_str = "\n---\n".join([f"[Fragmento]: {c.text}" for c in context_chunks])

        # Simulate a response that actually uses the context
        response_text = f"Based on the provided context, for the query '{query}', the following information was found:\n"
        for i, chunk in enumerate(context_chunks):
            response_text += f"\nContext Chunk {i+1} (ID: {chunk.id}):\n{chunk.text}\n"
        response_text += f"\n(Generated by {self.model_id} local based on {len(context_chunks)} chunks structured)"

        return response_text

# ==========================================
# 6. ORQUESTADOR MAESTRO Y MATRIZ DE EXPERIMENTACIÓN
# ==========================================
class RAGPipelineOrchestrator:
    def __init__(self, texto_limpio: str):
        self.texto_limpio = texto_limpio
        self.resultados: List[ExperimentoResultado] = []

    def ejecutar_matriz_completa(self, query: str):
        # Componentes definidos en tu matriz + Variaciones ultra-ligeras para CPU local
        chunking_options = {
            "C2.1_RECURSIVE": RecursiveCharacterChunking(),
            "C2.2_STRUCTURE_AWARE": StructureAwareChunking(),
            "C2.3_PARENT_CHILD": ParentChildChunking()
        }

        # Añadido un modelo ultra-pequeño (all-MiniLM) para máxima velocidad de embedding local
        embedding_options = [
            "sentence-transformers/all-MiniLM-L6-v2", # Extra-ligero (384 dim)
            "BAAI/bge-small-en-v1.5"                 # Tu matriz (int8 cuantizable local)
        ]

        retrieval_options = ["R4.1_COSINE", "R4.2_HYBRID_RRF"]
        reranker_options = ["None", "BAAI/bge-reranker-base"]

        # Añadidos modelos de 1.5B e incluso inferiores para entornos de bajísima VRAM
        slm_options = [
            "Qwen/Qwen2.5-1.5B-Instruct",             # Ultra-ligero de alta precisión financiera
            "microsoft/Phi-3-mini-4k-instruct",       # Tu matriz
            "meta-llama/Llama-3-8B-Instruct-Q4"       # Tu matriz (Límite local en hardware básico)
        ]

        # Cache de chunks e índices para evitar recalcular fases pesadas de la matriz
        chunks_cache = {}
        index_cache = {}

        # Generar el producto cartesiano de todas las combinaciones viables
        combinaciones = list(itertools.product(
            chunking_options.keys(),
            embedding_options,
            retrieval_options,
            reranker_options,
            slm_options
        ))

        print(f" Iniciando evaluación masiva local: {len(combinaciones)} permutaciones lógicas.")
        print("-" * 80)

        for idx, (c_key, emb_model, ret_strat, rerank_model, slm_model) in enumerate(combinaciones):
            config_id = f"EXP_{idx+1:03d}"
            start_time = time.time()

            # --- FASE 1 & 2: Chunking (con Reutilización de Caché) ---
            if c_key not in chunks_cache:
                chunks_cache[c_key] = chunking_options[c_key].execute(self.texto_limpio)
            chunks_actuales = chunks_cache[c_key]

            # --- FASE 3: Indexación Vectorial (con Reutilización de Caché por modelo/chunk) ---
            cache_index_key = f"{c_key}_{emb_model}"
            if cache_index_key not in index_cache:
                index_cache[cache_index_key] = VectorStoreIndex(emb_model, chunks_actuales)
            indice_actual = index_cache[cache_index_key]

            # --- FASE 4: Recuperación (Retrieval) ---
            engine = RetrievalEngine(indice_actual, chunks_actuales)
            candidatos_recuperados = engine.retrieve(query, strategy=ret_strat, top_k=20)

            # --- FASE 5: Re-ranking ---
            ranker = ReRanker(rerank_model)
            candidatos_finales = ranker.rerank(query, candidatos_recuperados, top_n=5)

            # --- FASE 6: Generación con SLM Local ---
            slm = SmallLanguageModel(slm_model)
            respuesta = slm.generate(query, candidatos_finales)

            # Registro de métricas e impactos de rendimiento
            elapsed_time = time.time() - start_time
            tokens_estimados = sum(len(c.text.split()) for c in candidatos_finales)

            resultado = ExperimentoResultado(
                config_id=config_id,
                chunk_strategy=c_key,
                embedding_model=emb_model.split("/")[-1],
                retrieval_strategy=ret_strat,
                reranker_model=rerank_model.split("/")[-1], # Fixed typo here
                slm_model=slm_model.split("/")[-1],
                tiempo_ejecucion_seg=round(elapsed_time, 4),
                respuesta=respuesta,
                tokens_recuperados=tokens_estimados
            )
            self.resultados.append(resultado)

    def mostrar_tabla_comparativa(self):
        # Muestra la comparativa de los experimentos en un formato scannable limpio
        headers = ["ID", "Chunking", "Embedding", "Retrieval", "ReRanker", "SLM", "Tiempo (s)", "Tokens Contexto"]
        tabla_datos = []

        # Ordenamos los resultados por menor tiempo de ejecución para identificar los flujos más competitivos
        resultados_ordenados = sorted(self.resultados, key=lambda x: x.tiempo_ejecucion_seg)

        for r in resultados_ordenados:
            tabla_datos.append([
                r.config_id, r.chunk_strategy, r.embedding_model,
                r.retrieval_strategy, r.reranker_model, r.slm_model,
                f"{r.tiempo_ejecucion_seg}s", r.tokens_recuperados
            ])

        print("\n=== MATRIZ DE COMPARATIVA DE RENDIMIENTO (ORDENADO POR VELOCIDAD LOCAL) ===")
        print(tabulate(tabla_datos, headers=headers, tablefmt="grid"))


# ==========================================
# 7. PUNTO DE EJECUCIÓN (PROBANDO EL SCRIPT MAESTRO)
# ==========================================
if __name__ == "__main__":
    # Tu variable con texto markdown limpio (Simulación orientada a informes 10-K)
    '''
    texto_limpio = """
    ## Item 1. Business
    We are a high-technology corporation specializing in cloud computing software solutions.

    ## Item 7. Management's Discussion and Analysis
    Our consolidated revenue increased by 15% year-over-year in fiscal year 2025.
    Net income was $4.5 billion compared to $3.9 billion in fiscal year 2024.
    The primary driver was our infrastructure-as-a-service (IaaS) division segment.

    ## Item 8. Financial Statements
    | Fiscal Year | Net Income (Billion) | Cash Flow (Billion) |
    |---|---|---|
    | 2025 | $4.5 | $5.1 |
    | 2024 | $3.9 | $4.2 |
    """
    '''

    texto_limpio = """
## Item 1. Business

### Overview
We are a global, high-technology corporation specializing in next-generation cloud computing software solutions, infrastructure architectures, and enterprise-grade software-as-a-service (SaaS) platforms. Founded with the mission to accelerate digital transformation, our comprehensive ecosystem enables enterprises, governments, and small-to-medium businesses (SMBs) to scale operations, optimize computational workloads, and leverage advanced artificial intelligence (AI) data analytics securely across hybrid and multi-cloud environments.

### Product and Service Offerings
Our business operations are organized into three primary high-growth segments:
* **Infrastructure-as-a-Service (IaaS):** Highly scalable compute, storage, and networking capabilities delivered through our proprietary global network of hyperscale data centers.
* **Platform-as-a-Service (PaaS):** Advanced developer tools, database management systems, and proprietary AI/Machine Learning frameworks that allow clients to build and deploy cloud-native applications efficiently.
* **Software-as-a-Service (SaaS):** Enterprise productivity tools, customer relationship management (CRM) software, and cybersecurity solutions integrated directly into our cloud architecture.

### Market and Competitive Landscape
The market for cloud computing solutions is intensely competitive, rapidly evolving, and subject to technological obsolescence. We compete primarily on the basis of platform reliability, data security, pricing elasticity, global latency, and the integration of cutting-edge AI features. Our competitors include legacy enterprise software vendors, specialized niche cloud providers, and other major hyperscale technology conglomerates.

---

## Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)

### Executive Overview of Fiscal Year 2025 Results
The following discussion should be read in conjunction with our consolidated financial statements and the related notes contained elsewhere in this Annual Report.

Our consolidated revenue increased by **15% year-over-year** in fiscal year 2025, driven by unprecedented enterprise adoption of our cloud-native architectures and a significant expansion in our multi-year customer commitments.

### Consolidated Results of Operations
* **Revenue Growth:** Total revenue expansion was primarily propelled by our **Infrastructure-as-a-Service (IaaS)** division segment, which experienced a 24% increase in workload migrations due to the rising demand for high-performance computing (HPC) and generative AI model training.
* **Net Income:** Net income for fiscal year 2025 reached **$4.5 billion**, a solid increase compared to **$3.9 billion** in fiscal year 2024. This profitability growth reflects improved operational leverage, optimized data center power efficiency, and strategic cost-management initiatives implemented across our global supply chain.
* **Operating Margins:** Gross margin improved by 120 basis points, largely attributable to economies of scale in infrastructure procurement, partially offset by increased research and development (R&D) investments in next-generation cybersecurity protocols.

### Liquidity and Capital Resources
As of December 31, 2025, we maintained a robust liquidity position with cash, cash equivalents, and short-term investments totaling $12.8 billion. We believe that our existing cash balances, combined with cash generated from ongoing operations, will be fully sufficient to satisfy our contractual obligations, planned capital expenditures for data center expansions, and strategic mergers and acquisitions for at least the next 12 months.

---

## Item 8. Financial Statements and Supplementary Data

### Consolidated Statements of Operations (Summary)
The following table sets forth a summary of our consolidated financial performance, highlighting key profitability metrics and operational cash flows for the fiscal years ended December 31, 2025, and December 31, 2024.

| Financial Metric | Fiscal Year 2025 | Fiscal Year 2024 | Year-over-Year Change (%) |
| :--- | :---: | :---: | :---: |
| **Total Consolidated Revenue** | $28.3 Billion | $24.6 Billion | +15.0% |
| **Operating Income** | $6.1 Billion | $5.2 Billion | +17.3% |
| **Net Income** | **$4.5 Billion** | **$3.9 Billion** | +15.4% |
| **Operating Cash Flow** | **$5.1 Billion** | **$4.2 Billion** | +21.4% |
| **Free Cash Flow** | $3.8 Billion | $3.1 Billion | +22.5% |

### Note 1: Basis of Presentation
The accompanying consolidated financial statements have been prepared in accordance with United States Generally Accepted Accounting Principles (U.S. GAAP). All intercompany accounts and transactions have been eliminated in consolidation. Certain prior-period amounts have been reclassified to conform to the current-period presentation.
"""

    query_test = "What was the Net Income in fiscal year 2025?"

    # Instanciación del Pipeline y ejecución completa en paralelo lógico
    pipeline = RAGPipelineOrchestrator(texto_limpio)
    pipeline.ejecutar_matriz_completa(query=query_test)

    # Presentación unificada de resultados financieros simulados
    pipeline.mostrar_tabla_comparativa()

 Iniciando evaluación masiva local: 72 permutaciones lógicas.
--------------------------------------------------------------------------------


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_831/1576625355.py:122: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_831/1576625355.py:122: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_831/1576625355.py:122: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_831/1576625355.py:122: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()



=== MATRIZ DE COMPARATIVA DE RENDIMIENTO (ORDENADO POR VELOCIDAD LOCAL) ===
+---------+----------------------+-------------------+-----------------+-------------------+------------------------+--------------+-------------------+
| ID      | Chunking             | Embedding         | Retrieval       | ReRanker          | SLM                    | Tiempo (s)   |   Tokens Contexto |
+=========+======================+===================+=================+===================+========================+==============+===================+
| EXP_052 | C2.3_PARENT_CHILD    | all-MiniLM-L6-v2  | R4.1_COSINE     | bge-reranker-base | Qwen2.5-1.5B-Instruct  | 0.018s       |               444 |
+---------+----------------------+-------------------+-----------------+-------------------+------------------------+--------------+-------------------+
| EXP_003 | C2.1_RECURSIVE       | all-MiniLM-L6-v2  | R4.1_COSINE     | None              | Llama-3-8B-Instruct-Q4 | 0.0181s      |               776 |
+----

In [ ]:
print("\n=== RESPUESTAS COMPLETAS POR EXPERIMENTO ===")
for r in pipeline.resultados:
    print(f"\nID: {r.config_id}")
    print(f"SLM Model: {r.slm_model}")
    print(f"Respuesta: {r.respuesta}")
    print("--------------------------------------------------")


=== RESPUESTAS COMPLETAS POR EXPERIMENTO ===

ID: EXP_001
SLM Model: Qwen2.5-1.5B-Instruct
Respuesta: Based on the provided context, for the query 'What was the Net Income in fiscal year 2025?', the following information was found:

Context Chunk 1 (ID: rec_3):
(Summary) The following table sets forth a summary of our consolidated financial performance, highlighting key profitability metrics and operational cash flows for the fiscal years ended December 31, 2025, and December 31, 2024. | Financial Metric | Fiscal Year 2025 | Fiscal Year 2024 | Year-over-Year Change (%) | | :--- | :---: | :---: | :---: | | **Total Consolidated Revenue** | $28.3 Billion | $24.6 Billion | +15.0% | | **Operating Income** | $6.1 Billion | $5.2 Billion | +17.3% | | **Net Income** | **$4.5 Billion** | **$3.9 Billion** | +15.4% | | **Operating Cash Flow** | **$5.1 Billion** | **$4.2 Billion** | +21.4% | | **Free Cash Flow** | $3.8 Billion | $3.1 Billion | +22.5% | ### Note 1: Basis of Presentation The accompa

In [ ]:
pipeline = RAGPipelineOrchestrator(texto_limpio)
pipeline.ejecutar_matriz_completa(query=query_test)
pipeline.mostrar_tabla_comparativa()

 Iniciando evaluación masiva local: 72 permutaciones lógicas.
--------------------------------------------------------------------------------


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_831/1576625355.py:122: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_831/1576625355.py:122: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_831/1576625355.py:122: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_831/1576625355.py:122: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()



=== MATRIZ DE COMPARATIVA DE RENDIMIENTO (ORDENADO POR VELOCIDAD LOCAL) ===
+---------+----------------------+-------------------+-----------------+-------------------+------------------------+--------------+-------------------+
| ID      | Chunking             | Embedding         | Retrieval       | ReRanker          | SLM                    | Tiempo (s)   |   Tokens Contexto |
+=========+======================+===================+=================+===================+========================+==============+===================+
| EXP_012 | C2.1_RECURSIVE       | all-MiniLM-L6-v2  | R4.2_HYBRID_RRF | bge-reranker-base | Llama-3-8B-Instruct-Q4 | 0.018s       |               776 |
+---------+----------------------+-------------------+-----------------+-------------------+------------------------+--------------+-------------------+
| EXP_006 | C2.1_RECURSIVE       | all-MiniLM-L6-v2  | R4.1_COSINE     | bge-reranker-base | Llama-3-8B-Instruct-Q4 | 0.0182s      |               776 |
+----

In [ ]:
from tabulate import tabulate

headers = ["ID", "SLM", "Respuesta", "Longitud Respuesta"]
tabla_respuestas = []

# Sort results by SLM model for better readability
resultados_ordenados_por_slm = sorted(pipeline.resultados, key=lambda x: x.slm_model)

for r in resultados_ordenados_por_slm:
    tabla_respuestas.append([
        r.config_id,
        r.slm_model,
        r.respuesta,
        len(r.respuesta)
    ])

print("\n=== MÉTRICAS DE RESPUESTAS GENERADAS ===")
print(tabulate(tabla_respuestas, headers=headers, tablefmt="grid"))


=== MÉTRICAS DE RESPUESTAS GENERADAS ===
+---------+------------------------+---------------------------------------------------------------------------------------------------+----------------------+
| ID      | SLM                    | Respuesta                                                                                         |   Longitud Respuesta |
+=========+========================+===================================================================================================+======================+
| EXP_003 | Llama-3-8B-Instruct-Q4 | [Respuesta Generada por meta-llama/Llama-3-8B-Instruct-Q4 local basado en 1 chunks estructurados] |                   97 |
+---------+------------------------+---------------------------------------------------------------------------------------------------+----------------------+
| EXP_006 | Llama-3-8B-Instruct-Q4 | [Respuesta Generada por meta-llama/Llama-3-8B-Instruct-Q4 local basado en 1 chunks estructurados] |                   97 

In [ ]:
import os
import time
import json
import itertools
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass, asdict
from tabulate import tabulate
from sentence_transformers import SentenceTransformer
import faiss

# Configuración simulada para ejecución local (Reemplazar con rutas o modelos reales)
# Nota: Para los SLMs usamos la interfaz de HuggingFace optimizada, pero se puede acoplar a llama.cpp
# indicando simplemente el cambio en la clase correspondiente.

# ==========================================
# 0. ESTRUCTURAS DE DATOS (DENTRO DEL PIPELINE)
# ==========================================

@dataclass
class Chunk:
    id: str
    text: str
    metadata: Dict[str, Any]

@dataclass
class ExperimentoResultado:
    config_id: str
    chunk_strategy: str
    embedding_model: str
    retrieval_strategy: str
    reranker_model: str
    slm_model: str
    tiempo_ejecucion_seg: float
    respuesta: str
    tokens_recuperados: int

# ==========================================
# 1. FASE DE CHUNKING (ESTRATEGIAS)
# ==========================================

class ChunkingStrategy:
    def execute(self, text: str) -> List[Chunk]:
        raise NotImplementedError

class RecursiveCharacterChunking(ChunkingStrategy):
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def execute(self, text: str) -> List[Chunk]:
        # Simulación de split recursivo ingenuo
        # En producción usar: from langchain.text_splitter import RecursiveCharacterTextSplitter
        chunks = []
        words = text.split()
        step = self.chunk_size // 5 # Estimación grosera de palabras
        overlap_step = self.chunk_overlap // 5

        idx = 0
        chunk_id = 0
        while idx < len(words):
            chunk_words = words[idx:idx+step]
            chunk_text = " ".join(chunk_words)
            chunks.append(Chunk(id=f"rec_{chunk_id}", text=chunk_text, metadata={"type": "recursive"}))
            idx += (step - overlap_step)
            chunk_id += 1
        return chunks

class StructureAwareChunking(ChunkingStrategy):
    def execute(self, text: str) -> List[Chunk]:
        # Fragmentación basada en encabezados Markdown (#, ##, ###)
        # Ideal para separar secciones macro de un 10-K sin mezclar tablas fiscales
        chunks = []
        sections = text.split("\n## ")
        for idx, section in enumerate(sections):
            if not section.strip():
                continue
            chunks.append(Chunk(id=f"struct_{idx}", text=f"## {section}", metadata={"type": "structure_aware"}))
        return chunks

class ParentChildChunking(ChunkingStrategy):
    def execute(self, text: str) -> List[Chunk]:
        # Divide en bloques grandes (Padre) y genera sub-fragmentos indexables (Hijos)
        # Devuelve los hijos con referencia cruzada en metadata al padre
        parent_sections = text.split("\n## ")
        chunks = []
        child_id_counter = 0

        for p_idx, p_text in enumerate(parent_sections):
            if not p_text.strip():
                continue
            parent_content = f"## {p_text}"
            # Generar hijos pequeños (ej. cada 2 párrafos)
            paragraphs = parent_content.split("\n\n")
            for c_idx in range(0, len(paragraphs), 2):
                child_text = "\n\n".join(paragraphs[c_idx:c_idx+2])
                if len(child_text.strip()) > 20:
                    chunks.append(Chunk(
                        id=f"child_{child_id_counter}",
                        text=child_text,
                        metadata={"type": "child", "parent_text": parent_content, "parent_id": f"parent_{p_idx}"}
                    ))
                    child_id_counter += 1
        return chunks

# ==========================================
# 2. FASE DE EMBEDDINGS E INDEXACIÓN (FAISS LOCAL)
# ==========================================
import numpy as np
try:
    import faiss
    from sentence_transformers import SentenceTransformer
except ImportError:
    # Fallback mock si corres en un entorno sin dependencias instaladas
    pass

class VectorStoreIndex:
    def __init__(self, model_name: str, chunks: List[Chunk]):
        self.model_name = model_name
        self.chunks = chunks
        # Para evitar sobrecargar la memoria en la permutación masiva, inicializamos bajo demanda
        self.model = SentenceTransformer(model_name)
        self.dimension = self.model.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatIP(self.dimension) # Inner Product para Similitud Coseno con vectores normalizados

        # Construir índice
        texts = [c.text for c in chunks]
        embeddings = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        # Normalizar para similitud coseno exacta
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)

    def search_vector(self, query: str, top_k: int = 50) -> List[Tuple[Chunk, float]]:
        query_emb = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_emb)
        distances, indices = self.index.search(query_emb, top_k)

        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx != -1:
                results.append((self.chunks[idx], float(dist)))
        return results

# ==========================================
# 3. FASE DE RECUPERACIÓN (RETRIEVAL HÍBRIDO & BM25)
# ==========================================
from rank_bm25 import BM25Okapi

class RetrievalEngine:
    def __init__(self, vector_index: VectorStoreIndex, chunks: List[Chunk]):
        self.vector_index = vector_index
        self.chunks = chunks
        # Inicializar BM25 léxico
        tokenized_corpus = [c.text.lower().split(" ") for c in chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def retrieve(self, query: str, strategy: str, top_k: int = 50) -> List[Chunk]:
        if strategy == "R4.1_COSINE":
            res = self.vector_index.search_vector(query, top_k=top_k)
            return [chunk for chunk, _ in res]

        elif strategy == "R4.2_HYBRID_RRF":
            # 1. Recuperación Densa
            dense_res = self.vector_index.search_vector(query, top_k=top_k)
            dense_ids = [c.id for c, _ in dense_res]

            # 2. Recuperación Léxica BM25
            tokenized_query = query.lower().split(" ")
            bm25_scores = self.bm25.get_scores(tokenized_query)
            top_bm25_indices = np.argsort(bm25_scores)[::-1][:top_k]
            bm25_ids = [self.chunks[idx].id for idx in top_bm25_indices]

            # 3. Fusión mediante Reciprocal Rank Fusion (RRF)
            rrf_scores = {}
            k_rrf = 60 # Constante estándar RRF

            for rank, cid in enumerate(dense_ids):
                rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k_rrf + rank + 1))
            for rank, cid in enumerate(bm25_ids):
                rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k_rrf + rank + 1))

            # Ordenar por el score fusionado
            sorted_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

            # Recomponer lista de Chunks ordenados
            chunks_dict = {c.id: c for c in self.chunks}
            final_chunks = []
            for cid, _ in sorted_ids:
                chunk_obj = chunks_dict[cid]
                # Modificación para la Ruta C: Parent-Child
                if chunk_obj.metadata.get("type") == "child":
                    # Si la estrategia requiere resolver el padre, devolvemos un chunk modificado con el contexto macro
                    parent_text = chunk_obj.metadata.get("parent_text", chunk_obj.text)
                    final_chunks.append(Chunk(id=chunk_obj.id, text=parent_text, metadata=chunk_obj.metadata))
                else:
                    final_chunks.append(chunk_obj)
            return final_chunks

        return []

# ==========================================
# 4. FASE DE RE-RANKING (CROSS-ENCODERS)
# ==========================================
class ReRanker:
    def __init__(self, model_name: str):
        self.model_name = model_name
        # Si es baseline "None", no cargamos modelo para ahorrar recursos
        if model_name != "None":
            # Simulación o inicialización de bge-reranker-base / local
            pass

    def rerank(self, query: str, candidates: List[Chunk], top_n: int = 5) -> List[Chunk]:
        if self.model_name == "None" or not candidates:
            return candidates[:top_n]

        # Simulación de ordenamiento Cross-Encoder ligero (bge-reranker-base)
        # En entorno real usar: from sentence_transformers import CrossEncoder
        # scores = model.predict([(query, c.text) for c in candidates])
        # Aquí simulamos basándonos en una ligera perturbación para emular cómputo local veloz
        return candidates[:top_n]

# ==========================================
# 5. FASE DE GENERACIÓN (MOCK / WRAPPER SLM LOCAL)
# ==========================================
class SmallLanguageModel:
    def __init__(self, model_id: str):
        self.model_id = model_id

    def generate(self, query: str, context_chunks: List[Chunk]) -> str:
        # Construcción de un prompt financiero estricto basado en anclaje semántico
        context_str = "\n---\n".join([f"[Fragmento]: {c.text}" for c in context_chunks])

        # Simulate a response that actually uses the context
        response_text = f"Based on the provided context, for the query '{query}', the following information was found:\n"
        for i, chunk in enumerate(context_chunks):
            response_text += f"\nContext Chunk {i+1} (ID: {chunk.id}):\n{chunk.text}\n"
        response_text += f"\n(Generated by {self.model_id} local based on {len(context_chunks)} chunks structured)"

        return response_text

# ==========================================
# 6. ORQUESTADOR MAESTRO Y MATRIZ DE EXPERIMENTACIÓN
# ==========================================
class RAGPipelineOrchestrator:
    def __init__(self, texto_limpio: str):
        self.texto_limpio = texto_limpio
        self.resultados: List[ExperimentoResultado] = []

    def ejecutar_matriz_completa(self, query: str):
        # Componentes definidos en tu matriz + Variaciones ultra-ligeras para CPU local
        chunking_options = {
            "C2.1_RECURSIVE": RecursiveCharacterChunking(),
            "C2.2_STRUCTURE_AWARE": StructureAwareChunking(),
            "C2.3_PARENT_CHILD": ParentChildChunking()
        }

        # Añadido un modelo ultra-pequeño (all-MiniLM) para máxima velocidad de embedding local
        embedding_options = [
            "sentence-transformers/all-MiniLM-L6-v2", # Extra-ligero (384 dim)
            "sentence-transformers/all-mpnet-base-v2",
            "BAAI/bge-small-en-v1.5",                 # Tu matriz (int8 cuantizable local)
            "BAAI/bge-large-en-v1.5",
            "BAAI/bge-base-en-v1.5",
            "thenlper/gte-large"
        ]

        retrieval_options = ["R4.1_COSINE", "R4.2_HYBRID_RRF"]
        reranker_options = ["None", "BAAI/bge-reranker-base"]

        # Añadidos modelos de 1.5B e incluso inferiores para entornos de bajísima VRAM
        slm_options = [
            "Qwen/Qwen2.5-1.5B-Instruct",             # Ultra-ligero de alta precisión financiera
            "microsoft/Phi-3-mini-4k-instruct",       # Tu matriz
            "meta-llama/Llama-3-8B-Instruct-Q4"       # Tu matriz (Límite local en hardware básico)
        ]

        # Cache de chunks e índices para evitar recalcular fases pesadas de la matriz
        chunks_cache = {}
        index_cache = {}

        # Generar el producto cartesiano de todas las combinaciones viables
        combinaciones = list(itertools.product(
            chunking_options.keys(),
            embedding_options,
            retrieval_options,
            reranker_options,
            slm_options
        ))

        print(f" Iniciando evaluación masiva local: {len(combinaciones)} permutaciones lógicas.")
        print("-" * 80)

        for idx, (c_key, emb_model, ret_strat, rerank_model, slm_model) in enumerate(combinaciones):
            config_id = f"EXP_{idx+1:03d}"
            start_time = time.time()

            # --- FASE 1 & 2: Chunking (con Reutilización de Caché) ---
            if c_key not in chunks_cache:
                chunks_cache[c_key] = chunking_options[c_key].execute(self.texto_limpio)
            chunks_actuales = chunks_cache[c_key]

            # --- FASE 3: Indexación Vectorial (con Reutilización de Caché por modelo/chunk) ---
            cache_index_key = f"{c_key}_{emb_model}"
            if cache_index_key not in index_cache:
                index_cache[cache_index_key] = VectorStoreIndex(emb_model, chunks_actuales)
            indice_actual = index_cache[cache_index_key]

            # --- FASE 4: Recuperación (Retrieval) ---
            engine = RetrievalEngine(indice_actual, chunks_actuales)
            candidatos_recuperados = engine.retrieve(query, strategy=ret_strat, top_k=20)

            # --- FASE 5: Re-ranking ---
            ranker = ReRanker(rerank_model)
            candidatos_finales = ranker.rerank(query, candidatos_recuperados, top_n=5)

            # --- FASE 6: Generación con SLM Local ---
            slm = SmallLanguageModel(slm_model)
            respuesta = slm.generate(query, candidatos_finales)

            # Registro de métricas e impactos de rendimiento
            elapsed_time = time.time() - start_time
            tokens_estimados = sum(len(c.text.split()) for c in candidatos_finales)

            resultado = ExperimentoResultado(
                config_id=config_id,
                chunk_strategy=c_key,
                embedding_model=emb_model.split("/")[-1],
                retrieval_strategy=ret_strat,
                reranker_model=rerank_model.split("/")[-1], # Fixed typo here
                slm_model=slm_model.split("/")[-1],
                tiempo_ejecucion_seg=round(elapsed_time, 4),
                respuesta=respuesta,
                tokens_recuperados=tokens_estimados
            )
            self.resultados.append(resultado)

    def mostrar_tabla_comparativa(self):
        # Muestra la comparativa de los experimentos en un formato scannable limpio
        headers = ["ID", "Chunking", "Embedding", "Retrieval", "ReRanker", "SLM", "Tiempo (s)", "Tokens Contexto"]
        tabla_datos = []

        # Ordenamos los resultados por menor tiempo de ejecución para identificar los flujos más competitivos
        resultados_ordenados = sorted(self.resultados, key=lambda x: x.tiempo_ejecucion_seg)

        for r in resultados_ordenados:
            tabla_datos.append([
                r.config_id, r.chunk_strategy, r.embedding_model,
                r.retrieval_strategy, r.reranker_model, r.slm_model,
                f"{r.tiempo_ejecucion_seg}s", r.tokens_recuperados
            ])

        print("\n=== MATRIZ DE COMPARATIVA DE RENDIMIENTO (ORDENADO POR VELOCIDAD LOCAL) ===")
        print(tabulate(tabla_datos, headers=headers, tablefmt="grid"))


# ==========================================
# 7. PUNTO DE EJECUCIÓN (PROBANDO EL SCRIPT MAESTRO)
# ==========================================
if __name__ == "__main__":
    # Tu variable con texto markdown limpio (Simulación orientada a informes 10-K)
    '''
    texto_limpio = """
    ## Item 1. Business
    We are a high-technology corporation specializing in cloud computing software solutions.

    ## Item 7. Management's Discussion and Analysis
    Our consolidated revenue increased by 15% year-over-year in fiscal year 2025.
    Net income was $4.5 billion compared to $3.9 billion in fiscal year 2024.
    The primary driver was our infrastructure-as-a-service (IaaS) division segment.

    ## Item 8. Financial Statements
    | Fiscal Year | Net Income (Billion) | Cash Flow (Billion) |
    |---|---|---|
    | 2025 | $4.5 | $5.1 |
    | 2024 | $3.9 | $4.2 |
    """
    '''

    texto_limpio = """
## Item 1. Business

### Overview
We are a global, high-technology corporation specializing in next-generation cloud computing software solutions, infrastructure architectures, and enterprise-grade software-as-a-service (SaaS) platforms. Founded with the mission to accelerate digital transformation, our comprehensive ecosystem enables enterprises, governments, and small-to-medium businesses (SMBs) to scale operations, optimize computational workloads, and leverage advanced artificial intelligence (AI) data analytics securely across hybrid and multi-cloud environments.

### Product and Service Offerings
Our business operations are organized into three primary high-growth segments:
* **Infrastructure-as-a-Service (IaaS):** Highly scalable compute, storage, and networking capabilities delivered through our proprietary global network of hyperscale data centers.
* **Platform-as-a-Service (PaaS):** Advanced developer tools, database management systems, and proprietary AI/Machine Learning frameworks that allow clients to build and deploy cloud-native applications efficiently.
* **Software-as-a-Service (SaaS):** Enterprise productivity tools, customer relationship management (CRM) software, and cybersecurity solutions integrated directly into our cloud architecture.

### Market and Competitive Landscape
The market for cloud computing solutions is intensely competitive, rapidly evolving, and subject to technological obsolescence. We compete primarily on the basis of platform reliability, data security, pricing elasticity, global latency, and the integration of cutting-edge AI features. Our competitors include legacy enterprise software vendors, specialized niche cloud providers, and other major hyperscale technology conglomerates.

---

## Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)

### Executive Overview of Fiscal Year 2025 Results
The following discussion should be read in conjunction with our consolidated financial statements and the related notes contained elsewhere in this Annual Report.

Our consolidated revenue increased by **15% year-over-year** in fiscal year 2025, driven by unprecedented enterprise adoption of our cloud-native architectures and a significant expansion in our multi-year customer commitments.

### Consolidated Results of Operations
* **Revenue Growth:** Total revenue expansion was primarily propelled by our **Infrastructure-as-a-Service (IaaS)** division segment, which experienced a 24% increase in workload migrations due to the rising demand for high-performance computing (HPC) and generative AI model training.
* **Net Income:** Net income for fiscal year 2025 reached **$4.5 billion**, a solid increase compared to **$3.9 billion** in fiscal year 2024. This profitability growth reflects improved operational leverage, optimized data center power efficiency, and strategic cost-management initiatives implemented across our global supply chain.
* **Operating Margins:** Gross margin improved by 120 basis points, largely attributable to economies of scale in infrastructure procurement, partially offset by increased research and development (R&D) investments in next-generation cybersecurity protocols.

### Liquidity and Capital Resources
As of December 31, 2025, we maintained a robust liquidity position with cash, cash equivalents, and short-term investments totaling $12.8 billion. We believe that our existing cash balances, combined with cash generated from ongoing operations, will be fully sufficient to satisfy our contractual obligations, planned capital expenditures for data center expansions, and strategic mergers and acquisitions for at least the next 12 months.

---

## Item 8. Financial Statements and Supplementary Data

### Consolidated Statements of Operations (Summary)
The following table sets forth a summary of our consolidated financial performance, highlighting key profitability metrics and operational cash flows for the fiscal years ended December 31, 2025, and December 31, 2024.

| Financial Metric | Fiscal Year 2025 | Fiscal Year 2024 | Year-over-Year Change (%) |
| :--- | :---: | :---: | :---: |
| **Total Consolidated Revenue** | $28.3 Billion | $24.6 Billion | +15.0% |
| **Operating Income** | $6.1 Billion | $5.2 Billion | +17.3% |
| **Net Income** | **$4.5 Billion** | **$3.9 Billion** | +15.4% |
| **Operating Cash Flow** | **$5.1 Billion** | **$4.2 Billion** | +21.4% |
| **Free Cash Flow** | $3.8 Billion | $3.1 Billion | +22.5% |

### Note 1: Basis of Presentation
The accompanying consolidated financial statements have been prepared in accordance with United States Generally Accepted Accounting Principles (U.S. GAAP). All intercompany accounts and transactions have been eliminated in consolidation. Certain prior-period amounts have been reclassified to conform to the current-period presentation.
"""

    query_test = "What was the Net Income in fiscal year 2025?"

    # Instanciación del Pipeline y ejecución completa en paralelo lógico
    pipeline = RAGPipelineOrchestrator(texto_limpio)
    pipeline.ejecutar_matriz_completa(query=query_test)

    # Presentación unificada de resultados financieros simulados
    pipeline.mostrar_tabla_comparativa()

In [ ]:
pip install langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Refactorización del Pipeline RAG con LangChain

He refactorizado el pipeline RAG para aprovechar la modularidad y extensibilidad que ofrece LangChain. Los cambios principales incluyen:

*   **Chunking (Fragmentación):** Se utiliza `RecursiveCharacterTextSplitter` de LangChain para manejar la división del texto en `Chunks`. Las estrategias `StructureAwareChunking` y `ParentChildChunking` ahora envuelven este splitter base y añaden lógica para pre-procesar el texto antes de la fragmentación de LangChain.
*   **Embeddings y Vector Store:** Se emplea `HuggingFaceEmbeddings` de `langchain_community.embeddings` para generar incrustaciones y `FAISS` de `langchain_community.vectorstores` como base de datos vectorial. Esto simplifica la creación y gestión del índice.
*   **Retrieval (Recuperación):** Ahora se obtiene un `retriever` directamente del `FAISS` Vector Store. La lógica de **RRF (Reciprocal Rank Fusion)** para la recuperación híbrida se mantiene como un paso posterior que combina resultados de búsquedas densas y léxicas.
*   **Re-ranking:** La clase `ReRanker` se ha adaptado para trabajar con los `Document` de LangChain. Aunque la lógica interna sigue siendo una simulación, ahora encaja mejor en un flujo de procesamiento de documentos de LangChain.
*   **Generación (SLM):** La `SmallLanguageModel` ahora utiliza `ChatPromptTemplate` de LangChain para construir el prompt, haciendo la gestión de prompts más robusta y compatible con la estructura de LangChain. La simulación de la respuesta se ha mantenido para el propósito de este benchmarking local.

Esta estructura permite intercambiar fácilmente componentes (como diferentes splitters, modelos de embedding o vector stores) siguiendo la interfaz de LangChain.

In [ ]:
import os
import time
import json
import itertools
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass, asdict
from tabulate import tabulate
import numpy as np

# Langchain imports
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# External library for BM25
from rank_bm25 import BM25Okapi

# ==========================================
# 0. ESTRUCTURAS DE DATOS (DENTRO DEL PIPELINE)
# ==========================================

@dataclass
class Chunk:
    id: str
    text: str
    metadata: Dict[str, Any]

@dataclass
class ExperimentoResultado:
    config_id: str
    chunk_strategy: str
    embedding_model: str
    retrieval_strategy: str
    reranker_model: str
    slm_model: str
    tiempo_ejecucion_seg: float
    respuesta: str
    tokens_recuperados: int

# ==========================================
# 1. FASE DE CHUNKING (ESTRATEGIAS) con LangChain
# ==========================================

class LangChainChunkingStrategy:
    def execute(self, text: str) -> List[Document]:
        raise NotImplementedError

class RecursiveCharacterChunkingLC(LangChainChunkingStrategy):
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            is_separator_regex=False,
        )

    def execute(self, text: str) -> List[Document]:
        # Langchain's RecursiveCharacterTextSplitter returns Documents directly
        docs = self.text_splitter.create_documents([text])
        # Add custom metadata (if needed, or map original Chunk.id to Document.metadata['id'])
        for i, doc in enumerate(docs):
            doc.metadata['id'] = f"rec_{i}"
            doc.metadata['type'] = 'recursive'
        return docs

class StructureAwareChunkingLC(LangChainChunkingStrategy):
    def execute(self, text: str) -> List[Document]:
        # Use MarkdownTextSplitter for structure awareness
        markdown_splitter = MarkdownTextSplitter(chunk_size=1000, chunk_overlap=200)
        docs = markdown_splitter.create_documents([text])
        for i, doc in enumerate(docs):
            doc.metadata['id'] = f"struct_{i}"
            doc.metadata['type'] = 'structure_aware'
        return docs

class ParentChildChunkingLC(LangChainChunkingStrategy):
    def execute(self, text: str) -> List[Document]:
        # This would typically involve a more complex Langchain setup like MultiVectorRetriever
        # For simplicity here, we'll simulate a basic parent-child effect by creating child chunks
        # and linking to a larger parent chunk in metadata, still using a basic splitter.

        # First, create larger 'parent' documents (e.g., by splitting less aggressively)
        parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=0)
        parent_docs = parent_splitter.create_documents([text])

        # Then, create smaller 'child' documents for indexing, linking them to their parent
        child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
        child_docs = []
        for p_idx, p_doc in enumerate(parent_docs):
            child_texts = child_splitter.split_text(p_doc.page_content)
            for c_idx, c_text in enumerate(child_texts):
                child_docs.append(Document(
                    page_content=c_text,
                    metadata={
                        "id": f"child_{p_idx}_{c_idx}",
                        "type": "child",
                        "parent_text": p_doc.page_content, # Store parent content in child metadata
                        "parent_id": f"parent_{p_idx}"
                    }
                ))
        return child_docs

# ==========================================
# 2. FASE DE EMBEDDINGS E INDEXACIÓN (FAISS LOCAL con LangChain)
# ==========================================

class VectorStoreIndexLC:
    def __init__(self, model_name: str, docs: List[Document]):
        self.model_name = model_name
        # Initialize HuggingFaceEmbeddings
        self.embeddings = HuggingFaceEmbeddings(model_name=model_name)

        # Create FAISS index from documents
        # FAISS.from_documents handles encoding and adding to index
        self.vectorstore = FAISS.from_documents(docs, self.embeddings)
        self.docs = docs # Store original documents for retrieval mapping

    def get_retriever(self, search_kwargs: Dict = None):
        if search_kwargs is None:
            search_kwargs = {"k": 50}
        return self.vectorstore.as_retriever(search_kwargs=search_kwargs)

# ==========================================
# 3. FASE DE RECUPERACIÓN (RETRIEVAL HÍBRIDO & BM25)
# ==========================================

class RetrievalEngineLC:
    def __init__(self, vector_index_lc: VectorStoreIndexLC, original_docs: List[Document]):
        self.vector_index_lc = vector_index_lc
        self.original_docs = original_docs

        # For BM25, we need the raw text content from the original documents
        tokenized_corpus = [doc.page_content.lower().split(" ") for doc in original_docs]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def retrieve(self, query: str, strategy: str, top_k: int = 50) -> List[Document]:
        if strategy == "R4.1_COSINE":
            # Use LangChain's vectorstore retriever directly
            retriever = self.vector_index_lc.get_retriever(search_kwargs={"k": top_k})
            return retriever.invoke(query)

        elif strategy == "R4.2_HYBRID_RRF":
            # 1. Dense Retrieval (using LangChain vectorstore)
            dense_retriever = self.vector_index_lc.get_retriever(search_kwargs={"k": top_k * 2}) # Retrieve more for RRF
            dense_results = dense_retriever.invoke(query)
            dense_ids = [doc.metadata.get('id', str(idx)) for idx, doc in enumerate(dense_results)]

            # 2. Lexical Retrieval BM25
            tokenized_query = query.lower().split(" ")
            bm25_scores = self.bm25.get_scores(tokenized_query)
            # Map BM25 scores back to original documents using their indices
            bm25_scored_docs = sorted([(score, self.original_docs[i]) for i, score in enumerate(bm25_scores)], key=lambda x: x[0], reverse=True)[:top_k * 2]
            bm25_ids = [doc.metadata.get('id', str(idx)) for idx, (_, doc) in enumerate(bm25_scored_docs)]

            # 3. RRF (Reciprocal Rank Fusion)
            rrf_scores = {}
            k_rrf = 60 # Standard RRF constant

            # Score dense results
            for rank, doc_id in enumerate(dense_ids):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank + 1))

            # Score BM25 results
            for rank, doc_id in enumerate(bm25_ids):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank + 1))

            # Sort by fused score and get top_k documents
            sorted_doc_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

            # Reconstruct list of Documents in final order
            # Need a map from ID to Document object. This is a simplification;
            # a real RAG would manage original docs or keep metadata consistent
            doc_map = {doc.metadata.get('id', str(i)): doc for i, doc in enumerate(self.original_docs)}
            final_docs = []
            for doc_id, _ in sorted_doc_ids:
                retrieved_doc = doc_map.get(doc_id)
                if retrieved_doc:
                    # For Parent-Child strategy, we might want to return the parent content
                    if retrieved_doc.metadata.get("type") == "child":
                        final_docs.append(Document(
                            page_content=retrieved_doc.metadata.get("parent_text", retrieved_doc.page_content),
                            metadata=retrieved_doc.metadata
                        ))
                    else:
                        final_docs.append(retrieved_doc)
            return final_docs

        return []

# ==========================================
# 4. FASE DE RE-RANKING (CROSS-ENCODERS)
# ==========================================
class ReRankerLC:
    def __init__(self, model_name: str):
        self.model_name = model_name
        # If using a real cross-encoder, you'd load it here, e.g:
        # self.cross_encoder = CrossEncoder(model_name)

    def rerank(self, query: str, candidates: List[Document], top_n: int = 5) -> List[Document]:
        if self.model_name == "None" or not candidates:
            return candidates[:top_n]

        # Simulación de ordenamiento Cross-Encoder
        # In a real scenario, you'd use self.cross_encoder.predict([(query, doc.page_content) for doc in candidates])
        # and then sort candidates based on scores.

        # For this simulation, we just return the top_n as is (or with a dummy sort)
        return candidates[:top_n]

# ==========================================
# 5. FASE DE GENERACIÓN (MOCK / WRAPPER SLM LOCAL con LangChain)
# ==========================================
class SmallLanguageModelLC:
    def __init__(self, model_id: str):
        self.model_id = model_id
        # Define a prompt template using LangChain
        self.prompt_template = ChatPromptTemplate.from_messages(
            [
                ("system", "You are an AI assistant specialized in financial analysis. Answer questions based ONLY on the provided context."),
                ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),
            ]
        )

    def generate(self, query: str, context_docs: List[Document]) -> str:
        context_str = "\n---\n".join([f"[Fragmento]: {doc.page_content}" for doc in context_docs])

        # Format the prompt using the template
        formatted_prompt = self.prompt_template.format_messages(context=context_str, question=query)

        # Simulate a response from the LLM
        response_text = f"Based on the provided context for the query '{query}':\n"
        for i, doc in enumerate(context_docs):
            response_text += f"\n[Context Document {i+1} (ID: {doc.metadata.get('id', 'N/A')})]\n{doc.page_content}\n"
        response_text += f"\n--- (Generated by {self.model_id} via LangChain mock) ---"

        return response_text

# ==========================================
# 6. ORQUESTADOR MAESTRO Y MATRIZ DE EXPERIMENTACIÓN
# ==========================================
class RAGPipelineOrchestratorLC:
    def __init__(self, texto_limpio: str):
        self.texto_limpio = texto_limpio
        self.resultados: List[ExperimentoResultado] = []

    def ejecutar_matriz_completa(self, query: str):
        chunking_options = {
            "C2.1_RECURSIVE": RecursiveCharacterChunkingLC(),
            "C2.2_STRUCTURE_AWARE": StructureAwareChunkingLC(),
            "C2.3_PARENT_CHILD": ParentChildChunkingLC()
        }

        embedding_options = [
            "sentence-transformers/all-MiniLM-L6-v2",
            "BAAI/bge-small-en-v1.5",
            "Qwen/Qwen3-Embedding-0.6B"
        ]

        retrieval_options = ["R4.1_COSINE", "R4.2_HYBRID_RRF"]
        reranker_options = ["None", "BAAI/bge-reranker-base"]

        slm_options = [
            "Qwen/Qwen2.5-1.5B-Instruct",
            "microsoft/Phi-3-mini-4k-instruct",
            "meta-llama/Llama-3-8B-Instruct-Q4"
        ]

        chunks_cache = {}
        index_cache = {}

        combinaciones = list(itertools.product(
            chunking_options.keys(),
            embedding_options,
            retrieval_options,
            reranker_options,
            slm_options
        ))

        print(f" Iniciando evaluación masiva local (LangChain): {len(combinaciones)} permutaciones lógicas.")
        print("-" * 80)

        for idx, (c_key, emb_model, ret_strat, rerank_model, slm_model) in enumerate(combinaciones):
            config_id = f"EXP_{idx+1:03d}"
            start_time = time.time()

            # --- FASE 1: Chunking (con Reutilización de Caché) ---
            if c_key not in chunks_cache:
                chunks_cache[c_key] = chunking_options[c_key].execute(self.texto_limpio)
            docs_actuales = chunks_cache[c_key]

            # --- FASE 2: Indexación Vectorial (con Reutilización de Caché por modelo/chunk) ---
            cache_index_key = f"{c_key}_{emb_model}"
            if cache_index_key not in index_cache:
                index_cache[cache_index_key] = VectorStoreIndexLC(emb_model, docs_actuales)
            indice_actual_lc = index_cache[cache_index_key]

            # --- FASE 3: Recuperación (Retrieval) ---
            engine = RetrievalEngineLC(indice_actual_lc, docs_actuales)
            candidatos_recuperados = engine.retrieve(query, strategy=ret_strat, top_k=20)

            # --- FASE 4: Re-ranking ---
            ranker = ReRankerLC(rerank_model)
            candidatos_finales = ranker.rerank(query, candidatos_recuperados, top_n=5)

            # --- FASE 5: Generación con SLM Local (LangChain-style) ---
            slm = SmallLanguageModelLC(slm_model)
            respuesta = slm.generate(query, candidatos_finales)

            # Registro de métricas e impactos de rendimiento
            elapsed_time = time.time() - start_time
            tokens_estimados = sum(len(doc.page_content.split()) for doc in candidatos_finales)

            resultado = ExperimentoResultado(
                config_id=config_id,
                chunk_strategy=c_key,
                embedding_model=emb_model.split("/")[-1],
                retrieval_strategy=ret_strat,
                reranker_model=rerank_model.split("/")[-1],
                slm_model=slm_model.split("/")[-1],
                tiempo_ejecucion_seg=round(elapsed_time, 4),
                respuesta=respuesta,
                tokens_recuperados=tokens_estimados
            )
            self.resultados.append(resultado)

    def mostrar_tabla_comparativa(self):
        headers = ["ID", "Chunking", "Embedding", "Retrieval", "ReRanker", "SLM", "Tiempo (s)", "Tokens Contexto"]
        tabla_datos = []

        resultados_ordenados = sorted(self.resultados, key=lambda x: x.tiempo_ejecucion_seg)

        for r in resultados_ordenados:
            tabla_datos.append([
                r.config_id, r.chunk_strategy, r.embedding_model,
                r.retrieval_strategy, r.reranker_model, r.slm_model,
                f"{r.tiempo_ejecucion_seg}s", r.tokens_recuperados
            ])

        print("\n=== MATRIZ DE COMPARATIVA DE RENDIMIENTO (ORDENADO POR VELOCIDAD LOCAL - LangChain) ===")
        print(tabulate(tabla_datos, headers=headers, tablefmt="grid"))


# ==========================================
# 7. PUNTO DE EJECUCIÓN (PROBANDO EL SCRIPT MAESTRO REFACTORIZADO)
# ==========================================
if __name__ == "__main__":
    texto_limpio = """
## Item 1. Business

### Overview
We are a global, high-technology corporation specializing in next-generation cloud computing software solutions, infrastructure architectures, and enterprise-grade software-as-a-service (SaaS) platforms. Founded with the mission to accelerate digital transformation, our comprehensive ecosystem enables enterprises, governments, and small-to-medium businesses (SMBs) to scale operations, optimize computational workloads, and leverage advanced artificial intelligence (AI) data analytics securely across hybrid and multi-cloud environments.

### Product and Service Offerings
Our business operations are organized into three primary high-growth segments:
* **Infrastructure-as-a-Service (IaaS):** Highly scalable compute, storage, and networking capabilities delivered through our proprietary global network of hyperscale data centers.
* **Platform-as-a-Service (PaaS):** Advanced developer tools, database management systems, and proprietary AI/Machine Learning frameworks that allow clients to build and deploy cloud-native applications efficiently.
* **Software-as-a-Service (SaaS):** Enterprise productivity tools, customer relationship management (CRM) software, and cybersecurity solutions integrated directly into our cloud architecture.

### Market and Competitive Landscape
The market for cloud computing solutions is intensely competitive, rapidly evolving, and subject to technological obsolescence. We compete primarily on the basis of platform reliability, data security, pricing elasticity, global latency, and the integration of cutting-edge AI features. Our competitors include legacy enterprise software vendors, specialized niche cloud providers, and other major hyperscale technology conglomerates.

---

## Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)

### Executive Overview of Fiscal Year 2025 Results
The following discussion should be read in conjunction with our consolidated financial statements and the related notes contained elsewhere in this Annual Report.

Our consolidated revenue increased by **15% year-over-year** in fiscal year 2025, driven by unprecedented enterprise adoption of our cloud-native architectures and a significant expansion in our multi-year customer commitments.

### Consolidated Results of Operations
* **Revenue Growth:** Total revenue expansion was primarily propelled by our **Infrastructure-as-a-Service (IaaS)** division segment, which experienced a 24% increase in workload migrations due to the rising demand for high-performance computing (HPC) and generative AI model training.
* **Net Income:** Net income for fiscal year 2025 reached **$4.5 billion**, a solid increase compared to **$3.9 billion** in fiscal year 2024. This profitability growth reflects improved operational leverage, optimized data center power efficiency, and strategic cost-management initiatives implemented across our global supply chain.
* **Operating Margins:** Gross margin improved by 120 basis points, largely attributable to economies of scale in infrastructure procurement, partially offset by increased research and development (R&D) investments in next-generation cybersecurity protocols.

### Liquidity and Capital Resources
As of December 31, 2025, we maintained a robust liquidity position with cash, cash equivalents, and short-term investments totaling $12.8 billion. We believe that our existing cash balances, combined with cash generated from ongoing operations, will be fully sufficient to satisfy our contractual obligations, planned capital expenditures for data center expansions, and strategic mergers and acquisitions for at least the next 12 months.

---

## Item 8. Financial Statements and Supplementary Data

### Consolidated Statements of Operations (Summary)
The following table sets forth a summary of our consolidated financial performance, highlighting key profitability metrics and operational cash flows for the fiscal years ended December 31, 2025, and December 31, 2024.

| Financial Metric | Fiscal Year 2025 | Fiscal Year 2024 | Year-over-Year Change (%) |
| :--- | :---: | :---: | :---: |
| **Total Consolidated Revenue** | $28.3 Billion | $24.6 Billion | +15.0% |
| **Operating Income** | $6.1 Billion | $5.2 Billion | +17.3% |
| **Net Income** | **$4.5 Billion** | **$3.9 Billion** | +15.4% |
| **Operating Cash Flow** | **$5.1 Billion** | **$4.2 Billion** | +21.4% |
| **Free Cash Flow** | $3.8 Billion | $3.1 Billion | +22.5% |

### Note 1: Basis of Presentation
The accompanying consolidated financial statements have been prepared in accordance with United States Generally Accepted Accounting Principles (U.S. GAAP). All intercompany accounts and transactions have been eliminated in consolidation. Certain prior-period amounts have been reclassified to conform to the current-period presentation.
"""

    query_test = "What was the Net Income in fiscal year 2025?"

    # Instanciación del Pipeline y ejecución completa en paralelo lógico
    pipeline_lc = RAGPipelineOrchestratorLC(texto_limpio)
    pipeline_lc.ejecutar_matriz_completa(query=query_test)

    # Presentación unificada de resultados financieros simulados
    pipeline_lc.mostrar_tabla_comparativa()

ModuleNotFoundError: No module named 'langchain_community'

### Evaluación de Respuestas Generadas y Contexto

Aquí se presentan las respuestas generadas por cada configuración del pipeline RAG refactorizado con LangChain, junto con la longitud de cada respuesta y la cantidad de tokens de contexto que se recuperaron. Esto nos ayuda a entender la verbosidad de cada modelo simulado y la cantidad de información que se le proporcionó para la generación.

In [ ]:
from tabulate import tabulate

headers = ["ID", "Chunking Strategy", "Embedding Model", "Retrieval Strategy", "ReRanker Model", "SLM Model", "Respuesta", "Longitud Respuesta (caracteres)", "Tokens Contexto Recuperados"]
tabla_respuestas = []

# Sort results by SLM model for better readability, then by chunking strategy
resultados_ordenados = sorted(pipeline_lc.resultados, key=lambda x: (x.slm_model, x.chunk_strategy))

for r in resultados_ordenados:
    tabla_respuestas.append([
        r.config_id,
        r.chunk_strategy,
        r.embedding_model,
        r.retrieval_strategy,
        r.reranker_model,
        r.slm_model,
        r.respuesta,
        len(r.respuesta),
        r.tokens_recuperados
    ])

print("\n=== DETALLE DE LAS RESPUESTAS Y CONTEXTO POR EXPERIMENTO (LangChain) ===")
print(tabulate(tabla_respuestas, headers=headers, tablefmt="grid"))


=== DETALLE DE LAS RESPUESTAS Y CONTEXTO POR EXPERIMENTO (LangChain) ===
+---------+----------------------+-------------------+----------------------+-------------------+------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------+-------------------------------+
| ID      | Chunking Strategy    | Embedding Model   | Retrieval Strategy   | ReRanker Model    | SLM Model              | Respuesta                                                                                                                                                                  

----------------

-------

---------------------


# PARTE BUENA


In [1]:
pip install sentence-transformers

In [2]:
pip install edgartools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 10.1 MB/s eta 0:00:00


In [3]:
from edgar import *
set_identity("your.name@example.com")

In [4]:
stock='AAPL'

In [5]:
stockdata = Company(stock)
filings = stockdata.get_filings(form="10-K")
latest_filing = filings[0]

filing = latest_filing

'''
# Get the full text of the filing
full_text = filing.text()
print(f"Full 10-K text: {stock} {len(full_text):,} characters ({len(full_text.split()):,} words)")
'''




'\n# Get the full text of the filing\nfull_text = filing.text()\nprint(f"Full 10-K text: {stock} {len(full_text):,} characters ({len(full_text.split()):,} words)")\n'

In [6]:
# Get a markdown representation
md = filing.markdown()


## Limpieza del texto

In [7]:
import re

def limpiar_numeros_linea(texto):
    # Patrón 1: Números solos o con punto final (ej: 48 o 48.)
    # Patrón 2: Formato de Apple (Nombre | Form | Número)
    patron = r"<div align='center'>(?:\d+\.?|\w+\sInc\.\s\|\s\d{4}\sForm\s10-K\s\|\s\d+)</div>"

    # Filtramos las líneas que NO coinciden con el patrón de "solo números"
    lineas = texto.split('\n')
    lineas_limpias = [l for l in lineas if not re.fullmatch(patron, l.strip())]

    return '\n'.join(lineas_limpias)

# Ejemplo de uso con tus datos
texto_limpio = limpiar_numeros_linea(md)
print(texto_limpio)

### UNITED STATES

### SECURITIES AND EXCHANGE COMMISSION

#### Washington, D.C. 20549

#### FORM 10-K
(Mark One)

<div align='center'>☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the fiscal year ended September 27, 2025

or

☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from to .</div>

#### Commission File Number:

#### 001-36743

#### Apple Inc.
<div align='center'>(Exact name of Registrant as specified in its charter)</div>

| California                               |     |                          94-2404110 |
| (State or other jurisdiction             
 of incorporation or organization)        |     | -I.R.S. Employer Identification No. |
| One Apple Park Way                       |     |                                     |
| Cupertino,California                     |     |                               95014 |
| (Address of principal executive offices) |     

In [8]:
pip install langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [8]:
pip install faiss-cpu # cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 65.9 MB/s eta 0:00:00


In [9]:
pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 12.2 MB/s eta 0:00:00


In [ ]:
import os
import time
import json
import itertools
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass, asdict
from tabulate import tabulate
import numpy as np
import pandas as pd # Import pandas for statistical analysis

# Langchain imports
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# External library for BM25
from rank_bm25 import BM25Okapi

# ==========================================
# 0. ESTRUCTURAS DE DATOS (DENTRO DEL PIPELINE)
# ==========================================

@dataclass
class Chunk:
    id: str
    text: str
    metadata: Dict[str, Any]

@dataclass
class ExperimentoResultado:
    config_id: str
    chunk_strategy: str
    embedding_model: str
    retrieval_strategy: str
    reranker_model: str
    slm_model: str
    tiempo_ejecucion_seg: float
    respuesta: str
    tokens_recuperados: int
    query: str # Added to store the query for each experiment

# ==========================================
# 1. FASE DE CHUNKING (ESTRATEGIAS) con LangChain
# ==========================================

class LangChainChunkingStrategy:
    def execute(self, text: str) -> List[Document]:
        raise NotImplementedError

class RecursiveCharacterChunkingLC(LangChainChunkingStrategy):
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            is_separator_regex=False,
        )

    def execute(self, text: str) -> List[Document]:
        # Langchain's RecursiveCharacterTextSplitter returns Documents directly
        docs = self.text_splitter.create_documents([text])
        # Add custom metadata (if needed, or map original Chunk.id to Document.metadata['id'])
        for i, doc in enumerate(docs):
            doc.metadata['id'] = f"rec_{i}"
            doc.metadata['type'] = 'recursive'
        return docs

class StructureAwareChunkingLC(LangChainChunkingStrategy):
    def execute(self, text: str) -> List[Document]:
        # Use MarkdownTextSplitter for structure awareness
        markdown_splitter = MarkdownTextSplitter(chunk_size=1000, chunk_overlap=200)
        docs = markdown_splitter.create_documents([text])
        for i, doc in enumerate(docs):
            doc.metadata['id'] = f"struct_{i}"
            doc.metadata['type'] = 'structure_aware'
        return docs

class ParentChildChunkingLC(LangChainChunkingStrategy):
    def execute(self, text: str) -> List[Document]:
        # This would typically involve a more complex Langchain setup like MultiVectorRetriever
        # For simplicity here, we'll simulate a basic parent-child effect by creating child chunks
        # and linking to a larger parent chunk in metadata, still using a basic splitter.

        # First, create larger 'parent' documents (e.g., by splitting less aggressively)
        parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=0)
        parent_docs = parent_splitter.create_documents([text])

        # Then, create smaller 'child' documents for indexing, linking them to their parent
        child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
        child_docs = []
        for p_idx, p_doc in enumerate(parent_docs):
            child_texts = child_splitter.split_text(p_doc.page_content)
            for c_idx, c_text in enumerate(child_texts):
                child_docs.append(Document(
                    page_content=c_text,
                    metadata={
                        "id": f"child_{p_idx}_{c_idx}",
                        "type": "child",
                        "parent_text": p_doc.page_content, # Store parent content in child metadata
                        "parent_id": f"parent_{p_idx}"
                    }
                ))
        return child_docs

# ==========================================
# 2. FASE DE EMBEDDINGS E INDEXACIÓN (FAISS LOCAL con LangChain)
# ==========================================

class VectorStoreIndexLC:
    def __init__(self, model_name: str, docs: List[Document]):
        self.model_name = model_name
        # Initialize HuggingFaceEmbeddings
        self.embeddings = HuggingFaceEmbeddings(model_name=model_name)

        # Create FAISS index from documents
        # FAISS.from_documents handles encoding and adding to index
        self.vectorstore = FAISS.from_documents(docs, self.embeddings)
        self.docs = docs # Store original documents for retrieval mapping

    def get_retriever(self, search_kwargs: Dict = None):
        if search_kwargs is None:
            search_kwargs = {"k": 50}
        return self.vectorstore.as_retriever(search_kwargs=search_kwargs)

# ==========================================
# 3. FASE DE RECUPERACIÓN (RETRIEVAL HÍBRIDO & BM25)
# ==========================================

class RetrievalEngineLC:
    def __init__(self, vector_index_lc: VectorStoreIndexLC, original_docs: List[Document]):
        self.vector_index_lc = vector_index_lc
        self.original_docs = original_docs

        # For BM25, we need the raw text content from the original documents
        tokenized_corpus = [doc.page_content.lower().split(" ") for doc in original_docs]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def retrieve(self, query: str, strategy: str, top_k: int = 50) -> List[Document]:
        if strategy == "R4.1_COSINE":
            # Use LangChain's vectorstore retriever directly
            retriever = self.vector_index_lc.get_retriever(search_kwargs={"k": top_k})
            return retriever.invoke(query)

        elif strategy == "R4.2_HYBRID_RRF":
            # 1. Dense Retrieval (using LangChain vectorstore)
            dense_retriever = self.vector_index_lc.get_retriever(search_kwargs={"k": top_k * 2}) # Retrieve more for RRF
            dense_results = dense_retriever.invoke(query)
            dense_ids = [doc.metadata.get('id', str(idx)) for idx, doc in enumerate(dense_results)]

            # 2. Lexical Retrieval BM25
            tokenized_query = query.lower().split(" ")
            bm25_scores = self.bm25.get_scores(tokenized_query)
            # Map BM25 scores back to original documents using their indices
            bm25_scored_docs = sorted([(score, self.original_docs[i]) for i, score in enumerate(bm25_scores)], key=lambda x: x[0], reverse=True)[:top_k * 2]
            bm25_ids = [doc.metadata.get('id', str(idx)) for idx, (_, doc) in enumerate(bm25_scored_docs)]

            # 3. RRF (Reciprocal Rank Fusion)
            rrf_scores = {}
            k_rrf = 60 # Standard RRF constant

            # Score dense results
            for rank, doc_id in enumerate(dense_ids):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank + 1))

            # Score BM25 results
            for rank, doc_id in enumerate(bm25_ids):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank + 1))

            # Sort by fused score and get top_k documents
            sorted_doc_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

            # Reconstruct list of Documents in final order
            # Need a map from ID to Document object. This is a simplification;
            # a real RAG would manage original docs or keep metadata consistent
            doc_map = {doc.metadata.get('id', str(idx)): doc for idx, doc in enumerate(self.original_docs)} # Fixed: using original_docs for doc_map
            final_docs = []
            for doc_id, _ in sorted_doc_ids:
                retrieved_doc = doc_map.get(doc_id)
                if retrieved_doc:
                    # For Parent-Child strategy, we might want to return the parent content
                    if retrieved_doc.metadata.get("type") == "child":
                        final_docs.append(Document(
                            page_content=retrieved_doc.metadata.get("parent_text", retrieved_doc.page_content),
                            metadata=retrieved_doc.metadata
                        ))
                    else:
                        final_docs.append(retrieved_doc)
            return final_docs

        return []

# ==========================================
# 4. FASE DE RE-RANKING (CROSS-ENCODERS)
# ==========================================
class ReRankerLC:
    def __init__(self, model_name: str):
        self.model_name = model_name
        # If using a real cross-encoder, you'd load it here, e.g:
        # self.cross_encoder = CrossEncoder(model_name)

    def rerank(self, query: str, candidates: List[Document], top_n: int = 5) -> List[Document]:
        if self.model_name == "None" or not candidates:
            return candidates[:top_n]

        # Simulación de ordenamiento Cross-Encoder
        # In a real scenario, you'd use self.cross_encoder.predict([(query, doc.page_content) for doc in candidates])
        # and then sort candidates based on scores.

        # For this simulation, we just return the top_n as is (or with a dummy sort)
        return candidates[:top_n]

# ==========================================
# 5. FASE DE GENERACIÓN (MOCK / WRAPPER SLM LOCAL con LangChain)
# ==========================================
class SmallLanguageModelLC:
    def __init__(self, model_id: str):
        self.model_id = model_id
        # Define a prompt template using LangChain
        self.prompt_template = ChatPromptTemplate.from_messages(
            [
                ("system", "You are an AI assistant specialized in financial analysis. Answer questions based ONLY on the provided context."),
                ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),
            ]
        )

    def generate(self, query: str, context_docs: List[Document]) -> str:
        context_str = "\n---\n".join([f"[Fragmento]: {doc.page_content}" for doc in context_docs])

        # Format the prompt using the template
        formatted_prompt = self.prompt_template.format_messages(context=context_str, question=query)

        # Simulate a response from the LLM
        response_text = f"Based on the provided context for the query '{query}':\n"
        for i, doc in enumerate(context_docs):
            response_text += f"\n[Context Document {i+1} (ID: {doc.metadata.get('id', 'N/A')})]\n{doc.page_content}\n"
        response_text += f"\n--- (Generated by {self.model_id} via LangChain mock) ---"

        return response_text

# ==========================================
# 6. ORQUESTADOR MAESTRO Y MATRIZ DE EXPERIMENTACIÓN
# ==========================================
class RAGPipelineOrchestratorLC:
    def __init__(self, texto_limpio: str):
        self.texto_limpio = texto_limpio
        self.resultados: List[ExperimentoResultado] = []

    def ejecutar_matriz_completa(self, query: str):
        chunking_options = {
            "C2.1_RECURSIVE": RecursiveCharacterChunkingLC(),
            "C2.2_STRUCTURE_AWARE": StructureAwareChunkingLC(),
            "C2.3_PARENT_CHILD": ParentChildChunkingLC()
        }

        embedding_options = [
            "sentence-transformers/all-MiniLM-L6-v2", # Extra-ligero (384 dim)
            #"sentence-transformers/all-mpnet-base-v2",
            "BAAI/bge-small-en-v1.5",                 # Tu matriz (int8 cuantizable local)
            #"BAAI/bge-base-en-v1.5",
            #"BAAI/bge-large-en-v1.5",
            #"thenlper/gte-large"
            "Qwen/Qwen3-Embedding-0.6B",
            "google/embeddinggemma-300m"
        ]

        retrieval_options = ["R4.1_COSINE", "R4.2_HYBRID_RRF"]
        reranker_options = ["None", "BAAI/bge-reranker-base"]

        slm_options = [
            "Qwen/Qwen2.5-1.5B-Instruct",
            "microsoft/Phi-3-mini-4k-instruct",
            #"meta-llama/Llama-3-8B-Instruct-Q4"
        ]

        chunks_cache = {}
        index_cache = {}

        combinaciones = list(itertools.product(
            chunking_options.keys(),
            embedding_options,
            retrieval_options,
            reranker_options,
            slm_options
        ))

        print(f" Iniciando evaluación masiva local (LangChain): {len(combinaciones)} permutaciones lógicas para la query: '{query}'.")
        print("-" * 80)

        for idx, (c_key, emb_model, ret_strat, rerank_model, slm_model) in enumerate(combinaciones):
            config_id = f"EXP_{idx+1:03d}"
            start_time = time.time()

            # --- FASE 1: Chunking (con Reutilización de Caché) ---
            if c_key not in chunks_cache:
                chunks_cache[c_key] = chunking_options[c_key].execute(self.texto_limpio)
            docs_actuales = chunks_cache[c_key]

            # --- FASE 2: Indexación Vectorial (con Reutilización de Caché por modelo/chunk) ---
            cache_index_key = f"{c_key}_{emb_model}"
            if cache_index_key not in index_cache:
                index_cache[cache_index_key] = VectorStoreIndexLC(emb_model, docs_actuales)
            indice_actual_lc = index_cache[cache_index_key]

            # --- FASE 3: Recuperación (Retrieval) ---
            engine = RetrievalEngineLC(indice_actual_lc, docs_actuales)
            candidatos_recuperados = engine.retrieve(query, strategy=ret_strat, top_k=20)

            # --- FASE 4: Re-ranking ---
            ranker = ReRankerLC(rerank_model)
            candidatos_finales = ranker.rerank(query, candidatos_recuperados, top_n=5)

            # --- FASE 5: Generación con SLM Local (LangChain-style) ---
            slm = SmallLanguageModelLC(slm_model)
            respuesta = slm.generate(query, candidatos_finales)

            # Registro de métricas e impactos de rendimiento
            elapsed_time = time.time() - start_time
            tokens_estimados = sum(len(doc.page_content.split()) for doc in candidatos_finales)

            resultado = ExperimentoResultado(
                config_id=config_id,
                chunk_strategy=c_key,
                embedding_model=emb_model.split("/")[-1],
                retrieval_strategy=ret_strat,
                reranker_model=rerank_model.split("/")[-1],
                slm_model=slm_model.split("/")[-1],
                tiempo_ejecucion_seg=round(elapsed_time, 4),
                respuesta=respuesta,
                tokens_recuperados=tokens_estimados,
                query=query # Storing the query in the result
            )
            self.resultados.append(resultado)

    def mostrar_tabla_comparativa(self):
        headers = ["ID", "Chunking", "Embedding", "Retrieval", "ReRanker", "SLM", "Tiempo (s)", "Tokens Contexto", "Query"]
        tabla_datos = []

        resultados_ordenados = sorted(self.resultados, key=lambda x: x.tiempo_ejecucion_seg)

        for r in resultados_ordenados:
            tabla_datos.append([
                r.config_id, r.chunk_strategy, r.embedding_model,
                r.retrieval_strategy, r.reranker_model, r.slm_model,
                f"{r.tiempo_ejecucion_seg}s", r.tokens_recuperados, r.query
            ])

        print("\n=== MATRIZ DE COMPARATIVA DE RENDIMIENTO (ORDENADO POR VELOCIDAD LOCAL - LangChain) ===")
        print(tabulate(tabla_datos, headers=headers, tablefmt="grid"))

# New function to display overall statistics
def display_overall_statistics(all_results: List[ExperimentoResultado]):
    if not all_results:
        print("No results to display statistics.")
        return

    # Convert results to a pandas DataFrame
    df = pd.DataFrame([asdict(r) for r in all_results])

    print("\n\n=== RESUMEN ESTADÍSTICO DE TIEMPOS DE EJECUCIÓN POR QUERY (Segundos) ===")
    # Group by query and calculate statistics
    stats_by_query = df.groupby('query')['tiempo_ejecucion_seg'].agg(['min', 'max', 'mean', 'median', 'std']).reset_index()
    print(tabulate(stats_by_query, headers='keys', tablefmt='grid'))

    print("\n\n=== RESUMEN ESTADÍSTICO GLOBAL DE TIEMPOS DE EJECUCIÓN (Segundos) ===")
    # Overall statistics
    overall_stats = df['tiempo_ejecucion_seg'].agg(['min', 'max', 'mean', 'median', 'std'])
    print(tabulate(overall_stats.to_frame().T, headers='keys', tablefmt='grid')) # Convert to DataFrame for tabulate


# ==========================================
# 7. PUNTO DE EJECUCIÓN (PROBANDO EL SCRIPT MAESTRO REFACTORIZADO)
# ==========================================
if __name__ == "__main__":

    # Removed hardcoded texto_limpio to use the one generated from 'md' in previous cells.
    # texto_limpio_OLD is kept as a reference but not used.
    texto_limpio_OLD = """
## Item 1. Business

### Overview
We are a global, high-technology corporation specializing in next-generation cloud computing software solutions, infrastructure architectures, and enterprise-grade software-as-a-service (SaaS) platforms. Founded with the mission to accelerate digital transformation, our comprehensive ecosystem enables enterprises, governments, and small-to-medium businesses (SMBs) to scale operations, optimize computational workloads, and leverage advanced artificial intelligence (AI) data analytics securely across hybrid and multi-cloud environments.

### Product and Service Offerings
Our business operations are organized into three primary high-growth segments:
*   **Infrastructure-as-a-Service (IaaS):** Highly scalable compute, storage, and networking capabilities delivered through our proprietary global network of hyperscale data centers.
*   **Platform-as-a-Service (PaaS):** Advanced developer tools, database management systems, and proprietary AI/Machine Learning frameworks that allow clients to build and deploy cloud-native applications efficiently.
*   **Software-as-a-Service (SaaS):** Enterprise productivity tools, customer relationship management (CRM) software, and cybersecurity solutions integrated directly into our cloud architecture.

### Market and Competitive Landscape
The market for cloud computing solutions is intensely competitive, rapidly evolving, and subject to technological obsolescence. We compete primarily on the basis of platform reliability, data security, pricing elasticity, global latency, and the integration of cutting-edge AI features. Our competitors include legacy enterprise software vendors, specialized niche cloud providers, and other major hyperscale technology conglomerates.

---

## Item 1A. Risk Factors

Our business is subject to a variety of risks. These include, but are not limited to, intense competition, rapid technological change, cybersecurity threats, and the ability to attract and retain qualified personnel. Economic downturns or adverse changes in economic conditions could also negatively impact our financial performance.

---

## Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)

### Executive Overview of Fiscal Year 2025 Results
The following discussion should be read in conjunction with our consolidated financial statements and the related notes contained elsewhere in this Annual Report.

Our consolidated revenue increased by **15% year-over-year** in fiscal year 2025, driven by unprecedented enterprise adoption of our cloud-native architectures and a significant expansion in our multi-year customer commitments.

### Consolidated Results of Operations
*   **Revenue Growth:** Total revenue expansion was primarily propelled by our **Infrastructure-as-a-Service (IaaS)** division segment, which experienced a 24% increase in workload migrations due to the rising demand for high-performance computing (HPC) and generative AI model training.
*   **Net Income:** Net income for fiscal year 2025 reached **$4.5 billion**, a solid increase compared to **$3.9 billion** in fiscal year 2024. This profitability growth reflects improved operational leverage, optimized data center power efficiency, and strategic cost-management initiatives implemented across our global supply chain.
*   **Operating Margins:** Gross margin improved by 120 basis points, largely attributable to economies of scale in infrastructure procurement, partially offset by increased research and development (R&D) investments in next-generation cybersecurity protocols.

### Liquidity and Capital Resources
As of December 31, 2025, we maintained a robust liquidity position with cash, cash equivalents, and short-term investments totaling $12.8 billion. We believe that our existing cash balances, combined with cash generated from ongoing operations, will be fully sufficient to satisfy our contractual obligations, planned capital expenditures for data center expansions, and strategic mergers and acquisitions for at least the next 12 months.

---

## Item 8. Financial Statements and Supplementary Data

### Consolidated Statements of Operations (Summary)
The following table sets forth a summary of our consolidated financial performance, highlighting key profitability metrics and operational cash flows for the fiscal years ended December 31, 2025, and December 31, 2024.

| Financial Metric | Fiscal Year 2025 | Fiscal Year 2024 | Year-over-Year Change (%) |
| :--- | :---: | :---: | :---: |
| **Total Consolidated Revenue** | $28.3 Billion | $24.6 Billion | +15.0% |
| **Operating Income** | $6.1 Billion | $5.2 Billion | +17.3% |
| **Net Income** | **$4.5 Billion** | **$3.9 Billion** | +15.4% |
| **Operating Cash Flow** | **$5.1 Billion** | **$4.2 Billion** | +21.4% |
| **Free Cash Flow** | $3.8 Billion | $3.1 Billion | +22.5% |

### Note 1: Basis of Presentation
The accompanying consolidated financial statements have been prepared in accordance with United States Generally Accepted Accounting Principles (U.S. GAAP). All intercompany accounts and transactions have been eliminated in consolidation. Certain prior-period amounts have been reclassified to conform to the current-period presentation.

### Note 2: Revenue Recognition
Revenue from contracts with customers is recognized when control of the promised goods or services is transferred to our customers, in an amount that reflects the consideration we expect to be entitled to in exchange for those goods or services. This typically occurs over time as services are rendered for SaaS and PaaS offerings, and at a point in time for certain IaaS transactions. Our primary revenue streams are derived from subscriptions to our cloud platforms, usage-based fees for infrastructure services, and professional services related to implementation and support.

### Note 3: Property and Equipment
Property and equipment are recorded at cost and depreciated over their estimated useful lives. Major categories include data center infrastructure, computer equipment, and leasehold improvements. Depreciation is primarily computed using the straight-line method. As of December 31, 2025, net property and equipment amounted to $18.7 billion, an increase from $16.2 billion in 2024, reflecting significant investments in expanding our global data center footprint.

### Note 4: Commitments and Contingencies
We are subject to various legal proceedings and claims that arise in the ordinary course of business. While the outcome of these matters cannot be predicted with certainty, we do not believe that the final outcome of any pending legal matters will have a material adverse effect on our financial position, results of operations, or cash flows.
"""


    query_tests = [
    # --- ESTRATEGIA Y PRODUCTO (BUSINESS & SEGMENTS) ---
    "What are the company's primary business segments and products?",
    "What is the percentage of revenue derived from each reportable business segment?",
    "How does the company describe its competitive position in the cloud computing and software-as-a-service (SaaS) markets?",
    ]

    query_tests_old = [
    # --- ESTRATEGIA Y PRODUCTO (BUSINESS & SEGMENTS) ---
    "What are the company's primary business segments and products?",
    "What is the percentage of revenue derived from each reportable business segment?",
    "How does the company describe its competitive position in the cloud computing and software-as-a-service (SaaS) markets?",
    "What are the primary subscription models and how does the company define customer churn or retention?",
    "What is the company's geographic revenue distribution, specifically highlighting exposure to the US, China, and Europe?",
    "Describe the company's strategy regarding the integration of Generative AI into its existing product suite.",
    "What are the key dependencies on third-party intellectual property or open-source software?",

    # --- FACTORES DE RIESGO (RISK FACTORS - ITEM 1A) ---
    "What are the key risk factors and uncertainties affecting the business?",
    "How does the company address risks related to cybersecurity breaches and the protection of user data?",
    "What are the specific risks mentioned regarding global semiconductor supply chains and hardware manufacturing?",
    "How could changes in antitrust laws or government regulations impact the company's market dominance?",
    "What are the risks associated with international operations, including currency fluctuations and geopolitical instability?",
    "Describe the risks of potential service interruptions in data centers or cloud infrastructure.",
    "What impact would a decline in advertising spend have on the company's revenue (if applicable)?",

    # --- RENDIMIENTO FINANCIERO (FINANCIAL PERFORMANCE) ---
    "Summarize the company's financial performance, including revenue and net income for the last fiscal year.",
    "What is the Year-over-Year (YoY) growth rate for the total revenue and the operating margin?",
    "How much did the company spend on Research and Development (R&D) as a percentage of total revenue?",
    "What is the total amount of cash, cash equivalents, and short-term investments held on the balance sheet?",
    "Describe the company's share repurchase (buyback) program and the total amount spent on dividends during the year.",
    "What is the company's effective tax rate and what are the main reasons for its fluctuation?",
    "Provide details on the company's long-term debt, including maturity dates and interest rate exposure.",

    # --- GOBERNANZA Y DIRECTIVA (GOVERNANCE) ---
    "Who are the principal executive officers and directors?",
    "What is the structure of executive compensation, including the balance between salary, bonuses, and stock options?",
    "What are the company's policies regarding insider trading and stock ownership for executives?",
    "Does the company have a dual-class share structure that gives founders or specific individuals disproportionate voting power?",
    "What are the stated ESG (Environmental, Social, and Governance) goals and how is progress being measured?",

    # --- INVERSIÓN Y PROYECTOS FUTUROS (CAPEX & MD&A) ---
    "What are the company's capital expenditures (CapEx) and investment plans for the upcoming fiscal year?",
    "How does management describe the intended use of cash for future acquisitions or strategic partnerships?",
    "What are the company's future unconditional purchase obligations, particularly for data center capacity or hardware components?",
    "Describe any significant investments in 'Other Bets' or non-core business areas mentioned in the MD&A.",

    # --- TEMAS LEGALES Y CONTABLES (LEGAL & ACCOUNTING) ---
    "Describe any significant legal proceedings or contingencies, particularly antitrust or patent infringement lawsuits.",
    "What are the critical accounting estimates that require the most significant management judgment?",
    "Has the company identified any material weaknesses in its internal control over financial reporting?",
    "Describe the impact of recently adopted accounting standards on the company's financial statements.",

    # --- MÉTRICAS ESPECÍFICAS DE TECH ---
    "What is the total headcount (number of employees) and how has it changed compared to the previous year?",
    "How much does the company pay in stock-based compensation (SBC) and what is its impact on GAAP net income?",
    "Does the company mention any reliance on specific AI hardware, such as NVIDIA GPUs or custom-designed chips?",
    "What is the company's backlog or remaining performance obligations (RPO) for long-term contracts?"
    ]


    # =============================================================================
    # RAG QUERY BANK — 10-K Filings (FY2025)
    # Empresas disponibles: AAPL, AMZN, AMD, AVGO, GOOGL, META, MSFT, TSLA
    # Nota: NVDA.md y MU.md no estaban en la carpeta al generarse este archivo.
    #
    # Leyenda de etiquetas en los comentarios de sección:
    #   [TODAS]    → pregunta con respuesta en los 10 documentos
    #   [ALGUNAS]  → pregunta con respuesta solo en ciertos documentos
    #               Se indica entre paréntesis las empresas esperadas
    #   [NINGUNA]  → pregunta diseñada para no tener respuesta directa en
    #               ninguno de los 10-K (p.ej. datos de competidores, años
    #               futuros, información no divulgada en 10-K)
    # =============================================================================

    query_tests_sp = [

    # =========================================================================
    # BLOQUE 1 — INFORMACIÓN CORPORATIVA BÁSICA [TODAS]
    # =========================================================================

    # --- Identificación y estructura legal ---
    "¿Cuál es el nombre legal exacto de la empresa y en qué estado está incorporada?",
    "¿Cuál es el número de identificación fiscal (EIN) de la empresa?",
    "¿Cuál es la dirección de la sede central (principal executive offices)?",
    "¿En qué mercado cotiza la empresa y cuál es su símbolo de ticker?",
    "¿Cuál es el número de archivo de la SEC (Commission File Number) de la empresa?",
    "¿Quién es el auditor externo y dónde está ubicado?",
    "¿Cuántas acciones comunes había en circulación a la fecha de cierre del ejercicio fiscal?",
    "¿Cuál fue la fecha de fin del ejercicio fiscal cubierto por este 10-K?",
    "¿Cuál fue la capitalización bursátil de las acciones en manos de no afiliados a mitad del año fiscal?",
    "¿La empresa es un 'well-known seasoned issuer' según la SEC?",

    # --- Estructura del informe ---
    "¿Qué partes componen este Form 10-K y qué contiene cada una?",
    "¿Qué documentos se incorporan por referencia en este 10-K?",
    "¿Tiene la empresa alguna filial consolidada que genere la mayor parte de los ingresos?",
    "¿Cuántos empleados tiene la empresa a nivel mundial al cierre del ejercicio?",
    "¿Cuántos empleados tiene la empresa en Estados Unidos específicamente?",


    # =========================================================================
    # BLOQUE 2 — MODELO DE NEGOCIO Y SEGMENTOS [TODAS]
    # =========================================================================

    "¿Cuáles son los principales segmentos de negocio reportados por la empresa?",
    "¿Cómo describe la empresa su modelo de negocio en el Item 1?",
    "¿Qué productos o servicios generan la mayor parte de los ingresos?",
    "¿Cuáles son los mercados geográficos más importantes para la empresa?",
    "¿Qué porcentaje de los ingresos totales proviene del mercado internacional?",
    "¿Qué porcentaje de los ingresos proviene de Estados Unidos?",
    "¿Cuáles son los principales clientes o tipo de clientes de la empresa?",
    "¿Tiene la empresa dependencia de algún cliente que represente más del 10% de ingresos?",
    "¿Cómo describe la empresa su ventaja competitiva o 'moat'?",
    "¿Cuáles son los principales competidores identificados en el 10-K?",
    "¿Cuál es la estrategia de crecimiento descrita por la dirección?",
    "¿La empresa opera en un mercado regulado y cuáles son las principales regulaciones aplicables?",
    "¿Qué papel juega la inteligencia artificial en la estrategia actual de la empresa?",
    "¿Qué inversiones en investigación y desarrollo (R&D) realizó la empresa en el año?",
    "¿Qué porcentaje de los ingresos representa el gasto en I+D?",


    # =========================================================================
    # BLOQUE 3 — RESULTADOS FINANCIEROS [TODAS]
    # =========================================================================

    # --- P&L ---
    "¿Cuáles fueron los ingresos totales (revenues) del último ejercicio fiscal?",
    "¿Cuál fue el beneficio bruto (gross profit) y el margen bruto?",
    "¿Cuáles fueron los gastos operativos totales (operating expenses)?",
    "¿Cuál fue el beneficio operativo (operating income) y el margen operativo?",
    "¿Cuál fue el beneficio neto (net income) del ejercicio?",
    "¿Cuál fue el beneficio por acción (EPS) básico y diluido?",
    "¿Cuál fue la tasa impositiva efectiva (effective tax rate)?",
    "¿Cuánto creció el ingreso total respecto al año anterior?",
    "¿Cuánto creció el beneficio neto respecto al año anterior?",
    "¿Cuáles fueron los gastos en ventas, generales y administrativos (SG&A)?",

    # --- Balance ---
    "¿Cuál fue el activo total (total assets) al cierre del ejercicio?",
    "¿Cuál fue el pasivo total (total liabilities)?",
    "¿Cuál fue el patrimonio neto (shareholders' equity)?",
    "¿Cuánto efectivo y equivalentes tenía la empresa al cierre del ejercicio?",
    "¿Cuál fue la deuda total a largo plazo (long-term debt)?",
    "¿Cuál fue el fondo de maniobra (working capital)?",
    "¿Cuál fue el ratio deuda neta / EBITDA?",
    "¿Qué inversiones a corto y largo plazo tenía la empresa en su balance?",
    "¿Cuánto representaban los activos intangibles y el fondo de comercio (goodwill) en el activo total?",

    # --- Flujo de caja ---
    "¿Cuál fue el flujo de caja operativo (operating cash flow)?",
    "¿Cuál fue el flujo de caja libre (free cash flow)?",
    "¿Cuánto invirtió la empresa en capex (capital expenditures)?",
    "¿Cuánto destinó la empresa a recompra de acciones propias (share buybacks)?",
    "¿Cuánto pagó la empresa en dividendos durante el ejercicio?",
    "¿Cuál fue el flujo de caja de financiación (financing cash flow)?",


    # =========================================================================
    # BLOQUE 4 — RETRIBUCIÓN AL ACCIONISTA [TODAS]
    # =========================================================================

    "¿Tiene la empresa programa de recompra de acciones? ¿Cuánto queda autorizado?",
    "¿Qué criterios usa la empresa para decidir el nivel de dividendos?",
    "¿Cuánto se incrementó el dividendo por acción respecto al año anterior?",
    "¿Cómo evoluciona el número de acciones diluidas a lo largo de los últimos tres años?",
    "¿Qué instrumentos de compensación basada en acciones (stock-based compensation) tiene la empresa?",
    "¿Cuánto registró la empresa en gastos de stock-based compensation?",


    # =========================================================================
    # BLOQUE 5 — RIESGOS [TODAS]
    # =========================================================================

    "¿Cuáles son los principales factores de riesgo (risk factors) descritos en el Item 1A?",
    "¿Qué riesgos macroeconómicos identifica la empresa como más relevantes?",
    "¿Qué riesgos relacionados con la cadena de suministro menciona la empresa?",
    "¿Cómo describe la empresa los riesgos de ciberseguridad y protección de datos?",
    "¿Qué riesgos regulatorios y de cumplimiento normativo se mencionan?",
    "¿Qué riesgos derivados de la competencia identifica la empresa?",
    "¿Qué riesgos relacionados con la inteligencia artificial se mencionan?",
    "¿Qué riesgos geopolíticos menciona la empresa (China, exportaciones, sanciones)?",
    "¿Qué riesgos climáticos o de sostenibilidad menciona el informe?",
    "¿Tiene la empresa riesgos de concentración de clientes o proveedores?",
    "¿Qué riesgos legales o litigios están pendientes según el Item 3?",
    "¿Menciona la empresa riesgos relacionados con la retención del talento?",
    "¿Cómo gestiona la empresa los riesgos de tipo de cambio (FX)?",
    "¿Qué riesgos de tasa de interés describe la empresa?",
    "¿Qué incidentes de ciberseguridad significativos ocurrieron durante el ejercicio?",


    # =========================================================================
    # BLOQUE 6 — CIBERSEGURIDAD (Item 1C) [TODAS]
    # =========================================================================

    "¿Qué proceso tiene la empresa para identificar y gestionar riesgos de ciberseguridad?",
    "¿Qué rol juega el consejo de administración en la supervisión de riesgos de ciberseguridad?",
    "¿Tiene la empresa un CISO o responsable de seguridad de la información identificado?",
    "¿Qué marcos de referencia de ciberseguridad (NIST, ISO, etc.) utiliza la empresa?",
    "¿Cómo evalúa la empresa a terceros proveedores desde el punto de vista de ciberseguridad?",


    # =========================================================================
    # BLOQUE 7 — GOBIERNO CORPORATIVO [TODAS]
    # =========================================================================

    "¿Quién es el CEO de la empresa según el 10-K?",
    "¿Quién es el CFO de la empresa?",
    "¿Cuántos miembros tiene el consejo de administración?",
    "¿Cuántos consejeros independientes hay en el consejo de administración?",
    "¿Tiene la empresa política de diversidad en el consejo de administración?",
    "¿Cuál es la remuneración total del CEO?",
    "¿Cuántos ejecutivos clave se identifican en la sección de información sobre ejecutivos?",
    "¿Qué comités tiene el consejo de administración (audit, compensation, nominating)?",


    # =========================================================================
    # BLOQUE 8 — PROPIEDADES E INFRAESTRUCTURA [TODAS]
    # =========================================================================

    "¿Cuáles son las principales propiedades e instalaciones de la empresa?",
    "¿Cuántos metros cuadrados de oficinas y centros de datos tiene la empresa?",
    "¿La empresa arrienda o es propietaria de sus principales instalaciones?",
    "¿Cuántos centros de datos opera la empresa o tiene en construcción?",


    # =========================================================================
    # BLOQUE 9 — CLOUD COMPUTING Y SERVICIOS DIGITALES
    # [ALGUNAS: AMZN, MSFT, GOOGL, META]
    # =========================================================================

    "¿Cuál fue el crecimiento en ingresos del negocio de nube (cloud) en el último año?",
    "¿Cuánto generó el segmento de nube en términos absolutos de ingresos?",
    "¿Cuál fue el margen operativo del segmento cloud?",
    "¿Qué cuota de mercado de cloud pública reclama o menciona la empresa?",
    "¿Cuáles son los principales servicios de nube ofrecidos y cuáles crecen más rápido?",
    "¿Cuánto invierte la empresa en infraestructura de centros de datos para cloud?",
    "¿Qué clientes empresariales significativos se mencionan en el segmento cloud?",
    "¿Cómo compite la empresa en cloud frente a sus rivales directos según el informe?",
    "¿Cuáles son los riesgos específicos del negocio cloud mencionados?",
    "¿Cómo impacta el negocio cloud en el margen consolidado de la empresa?",


    # =========================================================================
    # BLOQUE 10 — PUBLICIDAD DIGITAL
    # [ALGUNAS: META, GOOGL, AMZN]
    # =========================================================================

    "¿Cuánto representan los ingresos por publicidad del total de ingresos?",
    "¿Cuál fue el crecimiento de los ingresos publicitarios respecto al año anterior?",
    "¿Cómo describe la empresa el mecanismo de subasta o pricing de su publicidad?",
    "¿Qué métricas de usuarios activos reporta la empresa y cuál fue su evolución?",
    "¿Cuáles son los principales riesgos regulatorios sobre publicidad digital?",
    "¿Cómo afecta la política de privacidad (ATT de Apple, cookies de terceros) a los ingresos?",
    "¿Qué segmentos geográficos aportan más ingresos publicitarios?",
    "¿Cómo describe la empresa su plataforma de targeting publicitario?",


    # =========================================================================
    # BLOQUE 11 — SEMICONDUCTORES Y HARDWARE
    # [ALGUNAS: AMD, AVGO, TSLA]
    # =========================================================================

    "¿Qué líneas de productos de semiconductores describe la empresa?",
    "¿Cuáles son los principales mercados finales de los chips de la empresa?",
    "¿Cuánto representa la venta de semiconductores en el total de ingresos?",
    "¿Qué proceso de fabricación (nm) utilizan los chips más avanzados de la empresa?",
    "¿Fabrica la empresa sus chips internamente o los externaliza a foundries (fabless)?",
    "¿Qué restricciones de exportación de semiconductores afectan a la empresa?",
    "¿Cómo ha afectado el control de exportaciones a China a las ventas de la empresa?",
    "¿Qué nuevas arquitecturas de chip o roadmap tecnológico menciona la empresa?",
    "¿Cuáles son los riesgos de la cadena de suministro de semiconductores?",
    "¿Qué acuerdos de suministro con foundries (TSMC, Samsung, Intel Foundry) menciona la empresa?",
    "¿Cuál fue el capex en instalaciones de fabricación de semiconductores?",


    # =========================================================================
    # BLOQUE 12 — VEHÍCULOS ELÉCTRICOS Y AUTONOMÍA
    # [ALGUNAS: TSLA]
    # =========================================================================

    "¿Cuántos vehículos eléctricos entregó Tesla durante el ejercicio?",
    "¿Cuál fue el margen bruto del segmento Automotive de Tesla?",
    "¿Cuál fue el estado del desarrollo del Full Self-Driving (FSD) según el 10-K?",
    "¿Cuántos Superchargers o puntos de carga tenía Tesla desplegados al cierre?",
    "¿Qué plantas de fabricación ('Gigafactories') describe Tesla y en qué estado están?",
    "¿Cuánto generó el negocio de Energy Generation and Storage de Tesla?",
    "¿Qué ingresos generó el segmento Services and Other de Tesla?",
    "¿Cuál fue el nivel de inventario de vehículos de Tesla al cierre?",
    "¿Qué riesgos específicos de la industria automotriz menciona Tesla en su 10-K?",
    "¿Cómo describe Tesla el estado de desarrollo de su robot humanoide Optimus?",
    "¿Qué menciona Tesla sobre regulación de vehículos autónomos?",
    "¿Cuál es la política de precios de Tesla y cómo impactó en márgenes?",


    # =========================================================================
    # BLOQUE 13 — E-COMMERCE Y LOGÍSTICA
    # [ALGUNAS: AMZN]
    # =========================================================================

    "¿Cuánto generó el segmento North America de Amazon en ingresos?",
    "¿Cuánto generó el segmento International de Amazon en ingresos?",
    "¿Cuántos miembros tiene Amazon Prime y cuáles son los beneficios del programa?",
    "¿Cuáles son los principales riesgos logísticos y de cadena de suministro de Amazon?",
    "¿Qué menciona Amazon sobre su red de fulfillment centers y su automatización?",
    "¿Cuánto generó Amazon Web Services (AWS) en ingresos y cuál fue su margen?",
    "¿Cuál fue el crecimiento de AWS respecto al ejercicio anterior?",
    "¿Qué describe Amazon sobre su negocio de publicidad (Amazon Advertising)?",
    "¿Cuánto generó la publicidad de Amazon y cuál fue su tasa de crecimiento?",


    # =========================================================================
    # BLOQUE 14 — HARDWARE DE CONSUMO Y ECOSISTEMA
    # [ALGUNAS: AAPL, MSFT]
    # =========================================================================

    "¿Cuántos iPhone vendió Apple y cuáles fueron los ingresos del segmento iPhone?",
    "¿Cuál fue el ingreso del segmento Mac de Apple?",
    "¿Cuál fue el ingreso del segmento iPad de Apple?",
    "¿Cuál fue el ingreso del segmento Wearables, Home and Accessories de Apple?",
    "¿Cuánto generó el segmento Services de Apple y cuál fue su margen?",
    "¿Cuál es la estrategia de Apple en servicios de suscripción?",
    "¿Qué menciona Apple sobre su política de privacidad y su impacto competitivo?",
    "¿Cuántos usuarios tiene el ecosistema de dispositivos activos de Apple?",
    "¿Qué menciona Apple sobre Apple Intelligence y su estrategia de IA?",
    "¿Cómo describe Apple los riesgos regulatorios de la App Store?",
    "¿Cuáles son los ingresos de More Personal Computing (MPC) de Microsoft?",
    "¿Qué describe Microsoft sobre su negocio de Surface y Xbox?",


    # =========================================================================
    # BLOQUE 15 — SOFTWARE EMPRESARIAL Y PRODUCTIVIDAD
    # [ALGUNAS: MSFT, GOOGL, META]
    # =========================================================================

    "¿Cuánto generó el segmento Productivity and Business Processes de Microsoft?",
    "¿Cuál fue el crecimiento de los ingresos de Microsoft 365 (Office)?",
    "¿Cuántos suscriptores tiene Microsoft 365 en el segmento consumer y commercial?",
    "¿Cuánto generó el segmento Intelligent Cloud de Microsoft y cuál fue el margen?",
    "¿Cómo describe Microsoft su integración de Copilot en sus productos?",
    "¿Cuánto invirtió Microsoft en OpenAI y cómo describe esa relación?",
    "¿Cómo describe Google Workspace su crecimiento y base de usuarios enterprise?",
    "¿Qué menciona Meta sobre sus herramientas de negocio para empresas (Business Suite, Ads Manager)?",


    # =========================================================================
    # BLOQUE 16 — METAVERSO Y REALIDAD EXTENDIDA
    # [ALGUNAS: META, MSFT]
    # =========================================================================

    "¿Cuánto gastó Meta en el segmento Reality Labs y cuál fue su pérdida operativa?",
    "¿Qué productos de hardware XR (Quest, Ray-Ban) describe Meta en su 10-K?",
    "¿Cuál es la visión de Meta sobre el metaverso y cuándo prevé rentabilizar Reality Labs?",
    "¿Cuántos auriculares de realidad mixta vendió Meta?",
    "¿Qué menciona Microsoft sobre su negocio de realidad mixta (HoloLens, Mesh)?",


    # =========================================================================
    # BLOQUE 17 — INTELIGENCIA ARTIFICIAL (estrategia específica)
    # [ALGUNAS: MSFT, GOOGL, META, AMZN, AMD, AVGO, TSLA]
    # =========================================================================

    "¿Qué modelos de IA propios describe la empresa (Gemini, Copilot, Llama, etc.)?",
    "¿Cómo monetiza la empresa sus capacidades de IA en productos o servicios?",
    "¿Cuánto ha invertido la empresa en infraestructura de IA (GPUs, TPUs, aceleradores)?",
    "¿Qué acuerdos de licencia o partnership de IA menciona la empresa?",
    "¿Qué riesgos éticos y regulatorios de IA describe la empresa?",
    "¿Cómo describe la empresa el impacto de la IA en su productividad interna?",
    "¿Qué productos de IA generativa ha lanzado la empresa en el periodo reportado?",
    "¿Cómo posiciona la empresa su IA frente a competidores (OpenAI, Anthropic, etc.)?",
    "¿Cuánto destina la empresa a formación y retención de talento en IA?",
    "¿Describe la empresa su uso de IA para la moderación de contenido?",
    "¿Qué chips de IA o aceleradores diseña o usa la empresa?",  # AMD, AVGO, GOOGL (TPU)
    "¿Cuánto generó la venta de chips para centros de datos de IA?",  # AMD, AVGO


    # =========================================================================
    # BLOQUE 18 — REDES SOCIALES Y ENGAGEMENT
    # [ALGUNAS: META, GOOGL]
    # =========================================================================

    "¿Cuántos usuarios activos diarios (DAU/DAP) reportó Meta al cierre del ejercicio?",
    "¿Cuántos usuarios activos mensuales (MAU/MAP) reportó Meta?",
    "¿Cuál fue el ingreso promedio por usuario (ARPU) de Meta a nivel global?",
    "¿Cuánto tiempo medio pasa un usuario en las plataformas de Meta según el informe?",
    "¿Cómo describe Meta los riesgos de regulación de redes sociales (DSA, COPPA)?",
    "¿Qué menciona Meta sobre el impacto del algoritmo de recomendación en el engagement?",
    "¿Qué menciona Google sobre YouTube y sus métricas de uso?",
    "¿Cómo describe Alphabet el negocio de Google Search y su evolución?",
    "¿Qué menciona Alphabet sobre el impacto de la IA generativa en el negocio de búsqueda?",


    # =========================================================================
    # BLOQUE 19 — SEMICONDUCTOR DESIGN (AVGO específico)
    # [ALGUNAS: AVGO]
    # =========================================================================

    "¿Qué describe Broadcom sobre su negocio de infraestructura de software tras la adquisición de VMware?",
    "¿Cuánto generó el segmento de Semiconductor Solutions de Broadcom?",
    "¿Cuánto generó el segmento de Infrastructure Software de Broadcom?",
    "¿Cuál fue el impacto financiero de la integración de VMware en los resultados de Broadcom?",
    "¿Qué XPUs o aceleradores de IA custom describe Broadcom en su 10-K?",
    "¿Cuáles son los principales clientes de networking de Broadcom?",
    "¿Qué menciona Broadcom sobre sus acuerdos de suministro a hyperscalers?",
    "¿Cuál fue el ratio de apalancamiento de Broadcom tras la adquisición de VMware?",
    "¿Cuándo prevé Broadcom alcanzar sus objetivos de sinergias con VMware?",


    # =========================================================================
    # BLOQUE 20 — LITIGIOS Y ASUNTOS REGULATORIOS ESPECÍFICOS
    # [ALGUNAS]
    # =========================================================================

    # AAPL
    "¿Qué describe Apple sobre los litigios relacionados con la App Store y las disputas antimonopolio?",
    "¿Cuál es la situación del litigio de Apple con el DOJ o la Comisión Europea?",

    # GOOGL
    "¿Cuáles son los principales litigios antimonopolio contra Alphabet/Google descritos en el 10-K?",
    "¿Cómo describe Google el impacto de la sentencia DOJ sobre su negocio de búsqueda?",
    "¿Qué multas o sanciones regulatorias de la UE menciona Alphabet?",

    # META
    "¿Qué litigios de privacidad o violación del GDPR menciona Meta?",
    "¿Describe Meta el estado de las investigaciones del FTC sobre sus prácticas?",
    "¿Cómo describe Meta los riesgos regulatorios derivados del Digital Services Act (DSA)?",

    # MSFT
    "¿Qué menciona Microsoft sobre el escrutinio regulatorio de su adquisición de Activision?",
    "¿Qué litigios o investigaciones antimonopolio tiene abiertos Microsoft?",

    # AMZN
    "¿Qué litigios laborales o de competencia describe Amazon en su 10-K?",
    "¿Qué describe Amazon sobre las investigaciones regulatorias de su marketplace?",

    # TSLA
    "¿Qué litigios relacionados con el Autopilot o Full Self-Driving menciona Tesla?",
    "¿Qué investigaciones de NHTSA o agencias de seguridad menciona Tesla?",

    # AMD
    "¿Qué litigios de propiedad intelectual describe AMD en su 10-K?",


    # =========================================================================
    # BLOQUE 21 — SOSTENIBILIDAD Y ESG [TODAS]
    # =========================================================================

    "¿Qué compromisos de neutralidad de carbono o emisiones netas cero describe la empresa?",
    "¿Cuántas emisiones de CO2 (Scope 1, 2 y 3) declara la empresa?",
    "¿Qué porcentaje de energía renovable utiliza la empresa en sus operaciones?",
    "¿Qué programas de diversidad, equidad e inclusión (DEI) describe la empresa?",
    "¿Cuál es la política de bienestar de empleados descrita en el 10-K?",
    "¿Cuánto gasta la empresa en programas de responsabilidad social corporativa?",
    "¿Qué menciona la empresa sobre el riesgo climático en el contexto del TCFD?",
    "¿Qué iniciativas de economía circular o reducción de residuos electrónicos describe la empresa?",


    # =========================================================================
    # BLOQUE 22 — ADQUISICIONES Y M&A [ALGUNAS]
    # =========================================================================

    # MSFT
    "¿Qué adquisiciones relevantes completó Microsoft durante el ejercicio?",
    "¿Cómo describe Microsoft la integración de Activision Blizzard?",

    # AVGO
    "¿Qué adquisiciones describe Broadcom en el ejercicio y cuál fue el precio pagado?",

    # AMZN
    "¿Qué adquisiciones o inversiones estratégicas realizó Amazon durante el ejercicio?",
    "¿Cuánto invirtió Amazon en Anthropic y cómo describe esa relación?",

    # GOOGL
    "¿Qué adquisiciones realizó Alphabet y cuál fue el impacto en goodwill?",

    # META
    "¿Qué adquisiciones de startups de IA o XR realizó Meta?",

    # AMD
    "¿Qué adquisiciones realizó AMD y cómo impactaron en el segmento Data Center?",

    # AAPL
    "¿Cuánto gastó Apple en adquisiciones de empresas durante el ejercicio?",

    # TSLA
    "¿Realizó Tesla alguna adquisición estratégica durante el ejercicio?",


    # =========================================================================
    # BLOQUE 23 — CADENA DE SUMINISTRO Y FABRICACIÓN [ALGUNAS]
    # =========================================================================

    "¿Qué describe la empresa sobre la concentración geográfica de su cadena de suministro?",
    "¿Cuáles son los principales proveedores de componentes clave mencionados?",
    "¿Cómo gestiona la empresa el riesgo de escasez de componentes semiconductores?",
    "¿Qué impacto tuvieron los controles de exportación de EE.UU. sobre China en las ventas?",
    "¿Qué describe Apple sobre su dependencia de Foxconn y otros fabricantes?",   # AAPL
    "¿Describe AMD su plan de diversificación de foundries más allá de TSMC?",    # AMD
    "¿Qué describe Broadcom sobre el riesgo de concentración en su cadena de suministro?",  # AVGO
    "¿Cómo gestiona Tesla el suministro de baterías y celdas para sus vehículos?",  # TSLA


    # =========================================================================
    # BLOQUE 24 — DEUDA, CRÉDITO Y ESTRUCTURA DE CAPITAL [TODAS]
    # =========================================================================

    "¿Cuál es la deuda a largo plazo de la empresa y cuándo vencen las principales emisiones?",
    "¿Tiene la empresa líneas de crédito (revolving credit facilities) y en qué términos?",
    "¿Cuál es el rating crediticio de la empresa según las agencias mencionadas?",
    "¿Cómo describe la empresa su política de gestión de liquidez?",
    "¿Cuáles son los covenants o condiciones asociados a la deuda de la empresa?",
    "¿Qué emisiones de deuda (notas, bonos) realizó la empresa durante el ejercicio?",
    "¿Cuánto reembolsó la empresa en deuda durante el ejercicio?",
    "¿Cuánto pagó la empresa en intereses durante el ejercicio?",


    # =========================================================================
    # BLOQUE 25 — IMPUESTOS [TODAS]
    # =========================================================================

    "¿Cuál fue la provisión para impuestos sobre la renta del ejercicio?",
    "¿Cuáles son los principales elementos que explican la diferencia entre la tasa nominal y la efectiva?",
    "¿Tiene la empresa activos fiscales diferidos (deferred tax assets) significativos?",
    "¿Cuánto tiene la empresa en impuestos inciertos (uncertain tax positions)?",
    "¿Qué describe la empresa sobre el impacto del Pillar Two (OCDE) global minimum tax?",
    "¿En qué jurisdicciones fiscales tiene presencia significativa la empresa?",
    "¿Cuánto pagó la empresa en impuestos en efectivo (cash taxes paid)?",


    # =========================================================================
    # BLOQUE 26 — CONTROLES INTERNOS Y AUDITORÍA [TODAS]
    # =========================================================================

    "¿Cuál es la conclusión de la dirección sobre la efectividad del control interno sobre información financiera?",
    "¿Identificó el auditor externo alguna debilidad material (material weakness) en el periodo?",
    "¿Cómo describe la empresa su marco de control interno (COSO, SOX 404)?",
    "¿Cuántos partner de auditoría tiene asignados la empresa para su cuenta?",
    "¿Cuáles fueron los honorarios de auditoría y servicios relacionados pagados al auditor?",


    # =========================================================================
    # BLOQUE 27 — COMPENSACIÓN EJECUTIVA [TODAS]
    # =========================================================================

    "¿Cuáles son los componentes de la remuneración del CEO?",
    "¿Qué métricas se usan para calcular el bonus variable del equipo directivo?",
    "¿Cuántas acciones o RSU tiene concedidas el CEO?",
    "¿Cuál fue el ratio CEO-pay vs. mediana del empleado?",
    "¿Qué política de clawback (recuperación de compensación) tiene la empresa?",


    # =========================================================================
    # BLOQUE 28 — PREGUNTAS ESPECÍFICAS POR EMPRESA [ALGUNAS]
    # =========================================================================

    # AAPL — específicas
    "¿Cuál fue el volumen de ingresos de Apple en la región Greater China?",
    "¿Qué describe Apple sobre el estado de su chip M-series para Mac?",
    "¿Cuál fue el nivel de devolución de capital (dividendos + buybacks) de Apple durante el ejercicio?",
    "¿Cómo describe Apple el impacto de la regulación de la App Store en Europa tras el DMA?",

    # AMZN — específicas
    "¿Cuánto generó Amazon en ingresos por suscripciones (Prime, etc.)?",
    "¿Qué describe Amazon sobre el proyecto Kuiper (satélites de internet)?",
    "¿Cómo describe Amazon su inversión en Anthropic y su impacto en AWS AI?",
    "¿Cuántos centros de fulfillment tiene Amazon y en qué países?",

    # AMD — específicas
    "¿Cuánto generó el segmento Data Center de AMD y cuál fue su crecimiento?",
    "¿Cuánto generó el segmento Client (CPUs para PC) de AMD?",
    "¿Cuánto generó el segmento Gaming de AMD y por qué cayó?",
    "¿Cuánto generó el segmento Embedded de AMD?",
    "¿Qué describe AMD sobre sus GPUs Instinct (MI300X, MI350) y su posición en IA?",
    "¿Cuál fue el backlog o cartera de pedidos de AMD en Data Center?",

    # AVGO — específicas
    "¿Cuál fue el peso de los ingresos de custom AI accelerators (XPU) en el total de Broadcom?",
    "¿Cuánto contribuyó VMware a los ingresos de Infrastructure Software de Broadcom?",
    "¿Qué describe Broadcom sobre su relación con hyperscalers como Google y Meta?",
    "¿Cuándo vence la deuda más significativa de Broadcom tras la adquisición de VMware?",

    # GOOGL — específicas
    "¿Cuánto generó el segmento Google Cloud en ingresos y cuál fue su margen operativo?",
    "¿Qué describe Alphabet sobre su proyecto Waymo?",
    "¿Cuánto generó Google Search & other en ingresos?",
    "¿Cuánto generó YouTube ads en ingresos?",
    "¿Qué describe Alphabet sobre su modelo de IA Gemini y sus capacidades?",
    "¿Cuánto generó el segmento Other Bets de Alphabet y cuánto perdió?",
    "¿Cuántos empleados redujo Alphabet mediante reestructuración durante el ejercicio?",

    # META — específicas
    "¿Cuánto invirtió Meta en capex de infraestructura de IA?",
    "¿Cuál fue el número de usuarios activos diarios de WhatsApp?",
    "¿Cuánto generaron Instagram y Facebook en publicidad respectivamente?",
    "¿Cuál es la estrategia de Meta para monetizar WhatsApp?",
    "¿Qué describe Meta sobre Llama y su estrategia de IA open-source?",
    "¿Qué menciona Meta sobre la regulación de contenido de menores (KOSA, COPPA)?",

    # MSFT — específicas
    "¿Cuánto generó Azure y cuál fue su tasa de crecimiento en el trimestre final del ejercicio?",
    "¿Cuántos suscriptores tiene LinkedIn y cuánto genera en ingresos?",
    "¿Cuál fue el impacto de Copilot en el ARPU de Microsoft 365?",
    "¿Qué describe Microsoft sobre su participación en OpenAI y los riesgos asociados?",
    "¿Cuánto generó el negocio de gaming de Microsoft (Xbox, Activision) tras la integración?",
    "¿Cuánto destina Microsoft a capex de infraestructura de IA para el próximo ejercicio?",

    # TSLA — específicas
    "¿Cuántos modelos de vehículos tiene Tesla en producción activa?",
    "¿Cuál fue la tasa de utilización de las Gigafactories de Tesla?",
    "¿Qué menciona Tesla sobre el lanzamiento del Cybertruck y su rentabilidad?",
    "¿Cómo describe Tesla el negocio de Megapack y almacenamiento de energía?",
    "¿Qué menciona Tesla sobre el proyecto de taxi autónomo (Robotaxi/Cybercab)?",
    "¿Cuántos créditos regulatorios vendió Tesla y por qué importe?",
    "¿Cuál fue el gasto en I+D de Tesla como % de ventas?",


    # =========================================================================
    # BLOQUE 29 — PREGUNTAS COMPARATIVAS ENTRE EMPRESAS [TODAS/ALGUNAS]
    # =========================================================================

    "¿Qué empresa del grupo tiene el mayor margen bruto según su 10-K más reciente?",
    "¿Qué empresa del grupo tiene el mayor flujo de caja libre?",
    "¿Cuál es la empresa con mayor ratio de reinversión en I+D (R&D/ingresos)?",
    "¿Cuáles de estas empresas pagan dividendo y cuál tiene el mayor yield?",
    "¿Qué empresa tiene el mayor programa activo de recompra de acciones?",
    "¿Cuál de estas empresas tiene el mayor nivel de deuda neta?",
    "¿Cuál de estas empresas tiene el mayor número de empleados?",
    "¿Cuál de estas empresas describe el gasto de capex más elevado en términos absolutos?",
    "¿Cuáles de estas empresas tienen segmentos de cloud y cuál creció más rápido?",
    "¿Cuál es la empresa del grupo con mayor exposición a ingresos en China?",
    "¿Cuáles de estas empresas fabrican sus propios chips de silicio?",
    "¿Cuáles de estas empresas tienen un segmento de hardware de consumo relevante?",
    "¿Qué empresa tiene el mayor ratio precio-beneficio implícito según el EPS reportado?",
    "¿Cuál de estas empresas tiene la menor tasa impositiva efectiva?",
    "¿Cuáles de estas empresas mencionan el Pillar Two de la OCDE como riesgo fiscal relevante?",


    # =========================================================================
    # BLOQUE 30 — PREGUNTAS FUERA DE SCOPE / NINGUNA [NINGUNA]
    # Diseñadas para que el RAG NO encuentre respuesta en ninguno de los 10-K.
    # Útiles para medir alucinaciones o respuestas out-of-context.
    # =========================================================================

    # Empresas no incluidas en los documentos
    "¿Cuál fue el beneficio neto de NVIDIA en su último ejercicio fiscal?",
    "¿Cuánto generó Micron Technology (MU) en ingresos por HBM en el ejercicio?",
    "¿Cuál fue el ingreso de Intel en el segmento de Foundry Services?",
    "¿Cuántos vehículos eléctricos vendió BYD en 2025?",
    "¿Cuál fue el margen operativo de Samsung Semiconductor en 2025?",

    # Datos no divulgados en 10-K por naturaleza
    "¿Cuál será el precio de las acciones de Apple en diciembre de 2026?",
    "¿Qué empresa del grupo obtendrá mayor rentabilidad para el accionista en 2027?",
    "¿Cuánto pagará Microsoft en dividendos en el tercer trimestre de 2027?",
    "¿Cuál es la valoración privada de la división de IA de Google DeepMind de forma independiente?",
    "¿Cuánto gana personalmente Elon Musk de Tesla en salario bruto mensual?",

    # Información interna / no pública
    "¿Cuál es la estrategia de precios interna de AWS para grandes contratos enterprise?",
    "¿Qué dice el contrato de Apple con TSMC sobre precios de wafers?",
    "¿Cuánto pagó Meta en bonos discrecionales a ingenieros de IA en 2025?",
    "¿Cuál es el margen neto real de la App Store de Apple excluyendo costes de hosting?",
    "¿Cuántos despidos no declarados realizó Amazon en 2025 sin anuncio público?",

    # Datos de periodos no cubiertos
    "¿Cuánto ganó Microsoft en el primer trimestre de 2019?",
    "¿Cuál fue el beneficio de Google en 2010 antes de su escisión en Alphabet?",
    "¿Cuántos empleados tenía Amazon en 2015?",

    # Preguntas de dominio completamente ajeno
    "¿Cuál es la fórmula química del óxido de grafeno y sus propiedades eléctricas?",
    "¿Cuál es el resultado del partido de la Super Bowl del año 2025?",
    "¿Qué recomienda la OMS sobre el consumo de sal diario?",
    "¿Cuál fue el PIB de España en 2025?",
    "¿Cuál es la distancia media entre la Tierra y Marte?",

    ]

    query_tests_new = [

    # =========================================================================
    # BLOCK 1 — BASIC CORPORATE INFORMATION [ALL]
    # =========================================================================

    # --- Legal Structure and Identification ---
    "What is the exact legal name of the company and in which state is it incorporated?",
    "What is the company's Employer Identification Number (EIN)?",
    "What is the address of the company's principal executive offices?",
    "On which exchange is the company listed and what is its ticker symbol?",
    "What is the company's SEC Commission File Number?",
    "Who is the external auditor and where are they located?",
    "How many shares of common stock were outstanding as of the fiscal year-end closing date?",
    "What was the end date of the fiscal year covered by this Form 10-K?",
    "What was the aggregate market value of the voting and non-voting common equity held by non-affiliates as of the end of the second fiscal quarter?",
    "Is the company a 'well-known seasoned issuer' (WKSI) according to the SEC?",

    # --- Report Structure ---
    "What parts make up this Form 10-K and what does each part contain?",
    "What documents are incorporated by reference into this Form 10-K?",
    "Does the company have any consolidated subsidiaries that generate the majority of revenues?",
    "How many employees does the company have worldwide at fiscal year-end?",
    "How many employees does the company have specifically in the United States?",


    # =========================================================================
    # BLOCK 2 — BUSINESS MODEL AND SEGMENTS [ALL]
    # =========================================================================

    "What are the main business segments reported by the company?",
    "How does the company describe its business model in Item 1?",
    "Which products or services generate the majority of revenues?",
    "What are the company's most important geographic markets?",
    "What percentage of total revenues comes from the international market?",
    "What percentage of revenues originates from the United States?",
    "What are the company's primary customers or customer types?",
    "Is the company dependent on any single customer that accounts for more than 10% of revenues?",
    "How does the company describe its competitive advantage or 'moat'?",
    "Who are the main competitors identified in the Form 10-K?",
    "What is the growth strategy described by management?",
    "Does the company operate in a regulated market, and what are the main applicable regulations?",
    "What role does artificial intelligence play in the company's current strategy?",
    "What investments in research and development (R&D) did the company make during the year?",
    "What percentage of revenues does R&D spending represent?",


    # =========================================================================
    # BLOCK 3 — FINANCIAL RESULTS [ALL]
    # =========================================================================

    # --- P&L ---
    "What were the total revenues for the last fiscal year?",
    "What was the gross profit and the gross margin?",
    "What were the total operating expenses?",
    "What was the operating income and the operating margin?",
    "What was the net income for the fiscal year?",
    "What was the basic and diluted earnings per share (EPS)?",
    "What was the effective tax rate?",
    "How much did total revenue grow compared to the previous year?",
    "How much did net income grow compared to the previous year?",
    "What were the selling, general, and administrative (SG&A) expenses?",

    # --- Balance Sheet ---
    "What were the total assets at fiscal year-end?",
    "What were the total liabilities?",
    "What was the total shareholders' equity?",
    "How much cash and cash equivalents did the company hold at fiscal year-end?",
    "What was the total long-term debt?",
    "What was the working capital?",
    "What was the net debt-to-EBITDA ratio?",
    "What short-term and long-term investments did the company hold on its balance sheet?",
    "How much did intangible assets and goodwill represent out of total assets?",

    # --- Cash Flow Statement ---
    "What was the cash flow from operating activities (operating cash flow)?",
    "What was the free cash flow?",
    "How much did the company invest in capital expenditures (capex)?",
    "How much did the company allocate to share buybacks?",
    "How much did the company pay in dividends during the fiscal year?",
    "What was the cash flow from financing activities?",


    # =========================================================================
    # BLOCK 4 — SHAREHOLDER RETURN [ALL]
    # =========================================================================

    "Does the company have a share repurchase program? How much authorization remains?",
    "What criteria does the company use to determine dividend payout levels?",
    "How much did the dividend per share increase compared to the previous year?",
    "How has the number of diluted shares outstanding evolved over the past three years?",
    "What stock-based compensation instruments does the company utilize?",
    "How much did the company record in stock-based compensation expenses?",


    # =========================================================================
    # BLOCK 5 — RISK FACTORS [ALL]
    # =========================================================================

    "What are the primary risk factors described in Item 1A?",
    "What macroeconomic risks does the company identify as most relevant?",
    "What supply chain-related risks does the company mention?",
    "How does the company describe its cybersecurity and data protection risks?",
    "What regulatory and compliance risks are highlighted?",
    "What competition-driven risks does the company identify?",
    "What risks stemming from artificial intelligence are mentioned?",
    "What geopolitical risks does the company mention (e.g., China, exports, sanctions)?",
    "What climate or sustainability risks are mentioned in the report?",
    "Does the company face customer or supplier concentration risks?",
    "What legal proceedings or pending litigations are disclosed under Item 3?",
    "Does the company mention risks related to talent retention?",
    "How does the company manage foreign exchange (FX) rate risks?",
    "What interest rate risks does the company describe?",
    "What significant cybersecurity incidents occurred during the fiscal year?",


    # =========================================================================
    # BLOCK 6 — CYBERSECURITY (Item 1C) [ALL]
    # =========================================================================

    "What processes does the company have in place to identify and manage cybersecurity risks?",
    "What role does the board of directors play in overseeing cybersecurity risks?",
    "Does the company have an identified Chief Information Security Officer (CISO) or equivalent security lead?",
    "What cybersecurity frameworks (NIST, ISO, etc.) does the company utilize?",
    "How does the company evaluate third-party vendors from a cybersecurity perspective?",


    # =========================================================================
    # BLOCK 7 — CORPORATE GOVERNANCE [ALL]
    # =========================================================================

    "Who is the company's CEO according to the Form 10-K?",
    "Who is the company's CFO?",
    "How many members serve on the board of directors?",
    "How many independent directors are on the board of directors?",
    "Does the company have a board diversity policy?",
    "What is the total compensation of the CEO?",
    "How many executive officers are identified in the executive information section?",
    "What committees does the board of directors maintain (audit, compensation, nominating)?",


    # =========================================================================
    # BLOCK 8 — PROPERTIES AND INFRASTRUCTURE [ALL]
    # =========================================================================

    "What are the company's primary properties and facilities?",
    "How many square feet of office space and data centers does the company operate?",
    "Does the company lease or own its primary facilities?",
    "How many data centers does the company operate or currently have under construction?",


    # =========================================================================
    # BLOCK 9 — CLOUD COMPUTING AND DIGITAL SERVICES
    # [SOME: AMZN, MSFT, GOOGL, META]
    # =========================================================================

    "What was the revenue growth of the cloud business over the past year?",
    "How much revenue did the cloud segment generate in absolute terms?",
    "What was the operating margin of the cloud segment?",
    "What public cloud market share does the company claim or mention?",
    "What are the primary cloud services offered, and which ones are growing the fastest?",
    "How much does the company invest in data center infrastructure for cloud operations?",
    "What significant enterprise customers are mentioned in the cloud segment?",
    "How does the company compete in cloud services against its direct rivals according to the report?",
    "What specific risks related to the cloud business are highlighted?",
    "How does the cloud business impact the company's consolidated margin?",


    # =========================================================================
    # BLOCK 10 — DIGITAL ADVERTISING
    # [SOME: META, GOOGL, AMZN]
    # =========================================================================

    "How much do advertising revenues represent out of total revenues?",
    "What was the growth rate of advertising revenues compared to the previous year?",
    "How does the company describe its ad auction or pricing mechanisms?",
    "What active user metrics does the company report, and what was their trend?",
    "What are the main regulatory risks threatening digital advertising?",
    "How do privacy policy changes (e.g., Apple's ATT, third-party cookies) impact revenues?",
    "Which geographic segments contribute the most to advertising revenues?",
    "How does the company describe its ad targeting platform?",


    # =========================================================================
    # BLOCK 11 — SEMICONDUCTORS AND HARDWARE
    # [SOME: AMD, AVGO, TSLA]
    # =========================================================================

    "What semiconductor product lines does the company describe?",
    "What are the primary end markets for the company's chips?",
    "How much do semiconductor sales represent out of total revenues?",
    "What manufacturing process node (nm) is used for the company's most advanced chips?",
    "Does the company manufacture its chips internally, or does it outsource to external foundries (fabless)?",
    "What semiconductor export restrictions affect the company?",
    "How have export controls targeting China impacted the company's sales?",
    "What new chip architectures or technology roadmaps does the company mention?",
    "What are the main risks within the semiconductor supply chain?",
    "What supply agreements with foundries (e.g., TSMC, Samsung, Intel Foundry) are disclosed?",
    "What was the capital expenditure allocated to semiconductor manufacturing facilities?",


    # =========================================================================
    # BLOCK 12 — ELECTRIC VEHICLES AND AUTONOMY
    # [SOME: TSLA]
    # =========================================================================

    "How many electric vehicles did Tesla deliver during the fiscal year?",
    "What was the gross margin of Tesla's Automotive segment?",
    "What was the development status of Full Self-Driving (FSD) according to the Form 10-K?",
    "How many Superchargers or charging connectors did Tesla have deployed at year-end?",
    "What manufacturing plants ('Gigafactories') does Tesla describe, and what is their status?",
    "How much revenue did Tesla's Energy Generation and Storage business generate?",
    "What revenues were generated by Tesla's Services and Other segment?",
    "What was Tesla's vehicle inventory level at fiscal year-end?",
    "What automotive industry-specific risks does Tesla mention in its Form 10-K?",
    "How does Tesla describe the development status of its humanoid robot, Optimus?",
    "What does Tesla note regarding autonomous vehicle regulations?",
    "What is Tesla's pricing strategy, and how did it impact financial margins?",


    # =========================================================================
    # BLOCK 13 — E-COMMERCE AND LOGISTICS
    # [SOME: AMZN]
    # =========================================================================

    "How much revenue did Amazon's North America segment generate?",
    "How much revenue did Amazon's International segment generate?",
    "How many Amazon Prime members are there, and what are the program's benefits?",
    "What are Amazon's primary logistics and supply chain risks?",
    "What does Amazon mention regarding its network of fulfillment centers and its automation?",
    "How much revenue did Amazon Web Services (AWS) generate, and what was its margin?",
    "What was the growth rate of AWS compared to the previous fiscal year?",
    "What does Amazon disclose about its advertising business (Amazon Advertising)?",
    "How much revenue did Amazon's advertising segment generate, and what was its growth rate?",


    # =========================================================================
    # BLOCK 14 — CONSUMER HARDWARE AND ECOSYSTEM
    # [SOME: AAPL, MSFT]
    # =========================================================================

    "How many iPhones did Apple sell, and what were the revenues for the iPhone segment?",
    "What was the revenue for Apple's Mac segment?",
    "What was the revenue for Apple's iPad segment?",
    "What was the revenue for Apple's Wearables, Home, and Accessories segment?",
    "How much revenue did Apple's Services segment generate, and what was its margin?",
    "What is Apple's strategy regarding subscription services?",
    "What does Apple mention about its privacy policy and its competitive impact?",
    "How many users are in Apple's active installed base of devices?",
    "What does Apple state about Apple Intelligence and its AI strategy?",
    "How does Apple describe the regulatory risks threatening the App Store?",
    "What were Microsoft's revenues for More Personal Computing (MPC)?",
    "What does Microsoft disclose regarding its Surface and Xbox businesses?",


    # =========================================================================
    # BLOCK 15 — ENTERPRISE SOFTWARE AND PRODUCTIVITY
    # [SOME: MSFT, GOOGL, META]
    # =========================================================================

    "How much revenue did Microsoft's Productivity and Business Processes segment generate?",
    "What was the revenue growth rate for Microsoft 365 (Office)?",
    "How many Microsoft 365 subscribers are there in the consumer and commercial segments?",
    "How much revenue did Microsoft's Intelligent Cloud segment generate, and what was the margin?",
    "How does Microsoft describe its integration of Copilot across its product portfolio?",
    "How much has Microsoft invested in OpenAI, and how does it describe that relationship?",
    "How does Google Workspace describe its growth and enterprise user base?",
    "What does Meta mention about its business tools for enterprises (Business Suite, Ads Manager)?",


    # =========================================================================
    # BLOCK 16 — METAVERSE AND EXTENDED REALITY
    # [SOME: META, MSFT]
    # =========================================================================

    "How much did Meta spend on the Reality Labs segment, and what was its operating loss?",
    "What XR hardware products (Quest, Ray-Ban Meta smart glasses) does Meta describe in its Form 10-K?",
    "What is Meta's vision for the metaverse, and when does it expect Reality Labs to break even or become profitable?",
    "How many mixed reality headsets did Meta sell?",
    "What does Microsoft mention regarding its mixed reality business (HoloLens, Mesh)?",


    # =========================================================================
    # BLOCK 17 — ARTIFICIAL INTELLIGENCE (Specific Strategy)
    # [SOME: MSFT, GOOGL, META, AMZN, AMD, AVGO, TSLA]
    # =========================================================================

    "What proprietary AI models does the company describe (Gemini, Copilot, Llama, etc.)?",
    "How does the company monetize its AI capabilities within products or services?",
    "How much has the company invested in AI infrastructure (GPUs, TPUs, accelerators)?",
    "What AI licensing or partnership agreements does the company mention?",
    "What ethical and regulatory AI risks does the company outline?",
    "How does the company describe the impact of AI on its internal productivity?",
    "What generative AI products did the company launch during the reported period?",
    "How does the company position its AI capabilities against competitors (OpenAI, Anthropic, etc.)?",
    "How much capital does the company allocate to training and retaining AI talent?",
    "Does the company describe its use of AI for content moderation?",
    "What AI chips or accelerators does the company design or use?",
    "How much revenue was generated from the sale of AI data center chips?",


    # =========================================================================
    # BLOCK 18 — SOCIAL NETWORKS AND ENGAGEMENT
    # [SOME: META, GOOGL]
    # =========================================================================

    "How many Daily Active Users/People (DAU/DAP) did Meta report at fiscal year-end?",
    "How many Monthly Active Users/People (MAU/MAP) did Meta report?",
    "What was Meta's global Average Revenue Per User (ARPU)?",
    "What is the average time a user spends on Meta's platforms according to the report?",
    "How does Meta describe social media regulatory risks (DSA, COPPA)?",
    "What does Meta mention about the impact of recommendation algorithms on user engagement?",
    "What does Google mention about YouTube and its usage metrics?",
    "How does Alphabet describe the Google Search business and its performance?",
    "What does Alphabet mention about the impact of generative AI on the search business?",


    # =========================================================================
    # BLOCK 19 — SEMICONDUCTOR DESIGN (AVGO Specific)
    # [SOME: AVGO]
    # =========================================================================

    "What does Broadcom disclose about its infrastructure software business following the VMware acquisition?",
    "How much revenue did Broadcom's Semiconductor Solutions segment generate?",
    "How much revenue did Broadcom's Infrastructure Software segment generate?",
    "What was the financial impact of integrating VMware into Broadcom's results?",
    "What custom AI XPUs or accelerators does Broadcom describe in its Form 10-K?",
    "Who are Broadcom's primary networking customers?",
    "What does Broadcom mention about its supply agreements with hyperscalers?",
    "What was Broadcom's leverage ratio following the VMware acquisition?",
    "When does Broadcom expect to achieve its synergy targets with VMware?",


    # =========================================================================
    # BLOCK 20 — SPECIFIC LITIGATION AND REGULATORY MATTERS
    # [SOME]
    # =========================================================================

    # AAPL
    "What does Apple describe regarding antitrust disputes and litigation related to the App Store?",
    "What is the current status of Apple's litigation with the DOJ or the European Commission?",

    # GOOGL
    "What are the main antitrust lawsuits against Alphabet/Google described in the Form 10-K?",
    "How does Google describe the impact of the DOJ ruling on its search business?",
    "What EU regulatory fines or sanctions are mentioned by Alphabet?",

    # META
    "What privacy litigation or GDPR violation issues does Meta mention?",
    "Does Meta describe the status of FTC investigations into its practices?",
    "How does Meta describe the regulatory risks stemming from the Digital Services Act (DSA)?",

    # MSFT
    "What does Microsoft note about the regulatory scrutiny surrounding its Activision acquisition?",
    "What antitrust litigations or investigations does Microsoft currently face?",

    # AMZN
    "What labor or antitrust litigations does Amazon describe in its Form 10-K?",
    "What does Amazon disclose about regulatory investigations into its marketplace?",

    # TSLA
    "What litigations related to Autopilot or Full Self-Driving are mentioned by Tesla?",
    "What investigations by the NHTSA or other safety agencies are disclosed by Tesla?",

    # AMD
    "What intellectual property litigations does AMD describe in its Form 10-K?",


    # =========================================================================
    # BLOCK 21 — SUSTAINABILITY AND ESG [ALL]
    # =========================================================================

    "What carbon neutrality or net-zero emissions commitments does the company describe?",
    "How many CO2 emissions (Scope 1, 2, and 3) does the company report?",
    "What percentage of renewable energy does the company utilize across its operations?",
    "What diversity, equity, and inclusion (DEI) programs does the company outline?",
    "What employee well-being policy is described in the Form 10-K?",
    "How much does the company spend on corporate social responsibility programs?",
    "What does the company mention about climate risk in the context of the TCFD?",
    "What circular economy or electronic waste (e-waste) reduction initiatives does the company describe?",


    # =========================================================================
    # BLOCK 22 — ACQUISITIONS AND M&A [SOME]
    # =========================================================================

    # MSFT
    "What relevant acquisitions did Microsoft complete during the fiscal year?",
    "How does Microsoft describe the integration of Activision Blizzard?",

    # AVGO
    "What acquisitions does Broadcom describe for the fiscal year, and what was the purchase price paid?",

    # AMZN
    "What strategic acquisitions or investments did Amazon execute during the fiscal year?",
    "How much did Amazon invest in Anthropic, and how is that relationship described?",

    # GOOGL
    "What acquisitions did Alphabet complete, and what was the impact on goodwill?",

    # META
    "What acquisitions of AI or XR startups did Meta complete?",

    # AMD
    "What acquisitions did AMD execute, and how did they impact the Data Center segment?",

    # AAPL
    "How much capital did Apple deploy toward business acquisitions during the fiscal year?",

    # TSLA
    "Did Tesla make any strategic acquisitions during the fiscal year?",


    # =========================================================================
    # BLOCK 23 — SUPPLY CHAIN AND MANUFACTURING [SOME]
    # =========================================================================

    "What does the company disclose about the geographic concentration of its supply chain?",
    "Who are the main suppliers of critical components mentioned in the report?",
    "How does the company manage the risk of semiconductor component shortages?",
    "What impact did US export controls targeting China have on total sales?",
    "What does Apple disclose regarding its dependency on Foxconn and other manufacturers?",
    "Does AMD outline a diversification plan for foundries beyond TSMC?",
    "What does Broadcom state regarding concentration risks within its supply chain?",
    "How does Tesla manage the supply of batteries and cells for its vehicles?",


    # =========================================================================
    # BLOCK 24 — DEBT, CREDIT, AND CAPITAL STRUCTURE [ALL]
    # =========================================================================

    "What is the company's long-term debt, and when do the primary maturies fall due?",
    "Does the company maintain revolving credit facilities, and under what terms?",
    "What is the company's credit rating according to the credit rating agencies mentioned?",
    "How does the company describe its liquidity management policy?",
    "What are the financial covenants or conditions associated with the company's debt?",
    "What debt issuances (notes, bonds) did the company execute during the fiscal year?",
    "How much debt did the company repay during the fiscal year?",
    "How much interest expense did the company pay during the fiscal year?",


    # =========================================================================
    # BLOCK 25 — TAXES [ALL]
    # =========================================================================

    "What was the income tax provision for the fiscal year?",
    "What are the primary factors explaining the reconciliation between the statutory tax rate and the effective tax rate?",
    "Does the company hold significant deferred tax assets on its balance sheet?",
    "How much does the company hold in uncertain tax positions?",
    "What does the company disclose regarding the impact of the OECD Pillar Two global minimum tax?",
    "In which tax jurisdictions does the company maintain a significant presence?",
    "How much did the company pay in cash taxes during the fiscal year?",


    # =========================================================================
    # BLOCK 26 — INTERNAL CONTROLS AND AUDIT [ALL]
    # =========================================================================

    "What is management's conclusion regarding the effectiveness of internal control over financial reporting?",
    "Did the external auditor identify any material weaknesses during the period?",
    "How does the company describe its internal control framework (COSO, SOX 404)?",
    "How many audit partners are assigned by the firm to the company's account?",
    "What were the audit fees and related service fees paid to the external auditor?",


    # =========================================================================
    # BLOCK 27 — EXECUTIVE COMPENSATION [ALL]
    # =========================================================================

    "What are the core components of the CEO's compensation package?",
    "What financial or strategic metrics are used to calculate the executive leadership's variable bonus?",
    "How many stock options or RSUs have been granted to the CEO?",
    "What was the CEO-to-median-employee pay ratio?",
    "What clawback policy does the company maintain for executive compensation recovery?",


    # =========================================================================
    # BLOCK 28 — COMPANY-SPECIFIC QUESTIONS [SOME]
    # =========================================================================

    # AAPL — Specific
    "What was Apple's net revenue within the Greater China region?",
    "What does Apple disclose regarding the status and roadmap of its M-series Mac chips?",
    "What was Apple's total capital return (dividends + share buybacks) during the fiscal year?",
    "How does Apple describe the impact of App Store regulations in Europe under the DMA?",

    # AMZN — Specific
    "How much revenue did Amazon generate from subscription services (Prime, etc.)?",
    "What does Amazon disclose about Project Kuiper (satellite internet deployment)?",
    "How does Amazon describe its investment in Anthropic and its impact on AWS AI?",
    "How many fulfillment centers does Amazon operate, and in which countries?",

    # AMD — Specific
    "What were the revenues for AMD's Data Center segment, and what was its growth rate?",
    "How much revenue did AMD's Client segment (PC CPUs) generate?",
    "How much revenue did AMD's Gaming segment generate, and what caused its decline?",
    "How much revenue did AMD's Embedded segment generate?",
    "What does AMD state about its Instinct GPUs (MI300X, MI350) and its positioning in AI?",
    "What was AMD's order backlog for the Data Center segment?",

    # AVGO — Specific
    "What percentage of Broadcom's total revenue did custom AI accelerators (XPUs) represent?",
    "How much did VMware contribute to Broadcom's Infrastructure Software segment revenues?",
    "How does Broadcom describe its relationship with hyperscalers like Google and Meta?",
    "When does Broadcom's most significant debt tranche mature following the VMware acquisition?",

    # GOOGL — Specific
    "How much revenue did the Google Cloud segment generate, and what was its operating margin?",
    "What does Alphabet disclose regarding its Waymo autonomous vehicle project?",
    "How much revenue did 'Google Search & other' generate?",
    "How much revenue did YouTube ads generate?",
    "What does Alphabet share regarding its Gemini AI model and its core capabilities?",
    "How much revenue did Alphabet's 'Other Bets' segment generate, and what was its operating loss?",
    "How many employees did Alphabet lay off through restructuring programs during the year?",

    # META — Specific
    "How much capital expenditure did Meta allocate to AI infrastructure?",
    "What was the daily active user count for WhatsApp?",
    "How much revenue did Instagram and Facebook generate from advertising, respectively?",
    "What is Meta's commercial strategy for monetizing WhatsApp?",
    "What does Meta disclose about Llama and its open-source AI framework strategy?",
    "What does Meta mention regarding the regulation of content for minors (KOSA, COPPA)?",

    # MSFT — Specific
    "What was Azure's revenue and growth rate in the final quarter of the fiscal year?",
    "How many subscribers does LinkedIn have, and how much revenue does it generate?",
    "What was the financial impact of Copilot on Microsoft 365 ARPU?",
    "How does Microsoft describe its equity stake in OpenAI and the associated financial risks?",
    "How much revenue did Microsoft's gaming business (Xbox, Activision) generate post-integration?",
    "How much capital expenditure is Microsoft budgeting for AI infrastructure in the next fiscal year?",

    # TSLA — Specific
    "How many vehicle models does Tesla have in active production?",
    "What was the production capacity utilization rate across Tesla's Gigafactories?",
    "What does Tesla state about the launch of the Cybertruck and its path to profitability?",
    "How does Tesla describe the performance of its Megapack and energy storage business?",
    "What updates does Tesla provide on the autonomous taxi project (Robotaxi/Cybercab)?",
    "How many regulatory credits did Tesla sell, and what was the total proceeds?",
    "What was Tesla's R&D expenditure as a percentage of total sales?",


    # =========================================================================
    # BLOCK 29 — COMPARATIVE QUESTIONS ACROSS COMPANIES [ALL/SOME]
    # =========================================================================

    "Which company in the peer group reported the highest gross margin in its most recent Form 10-K?",
    "Which company in the peer group generated the highest free cash flow?",
    "Which company has the highest R&D reinvestment rate (R&D-to-revenue ratio)?",
    "Which of these companies pay dividends, and which one offers the highest dividend yield?",
    "Which company maintains the largest active share repurchase program?",
    "Which of these companies carries the highest level of net debt?",
    "Which of these companies has the largest headcount or employee base?",
    "Which of these companies reported the highest absolute capital expenditure (capex)?",
    "Which of these companies report a dedicated cloud business segment, and which one grew the fastest?",
    "Which company within the group has the highest revenue exposure to China?",
    "Which of these companies manufacture or design their own custom silicon chips?",
    "Which of these companies possess a meaningful consumer hardware business segment?",
    "Which company commands the highest implied price-to-earnings (P/E) ratio based on reported EPS?",
    "Which of these companies recorded the lowest effective tax rate?",
    "Which of these companies explicitly note the OECD Pillar Two initiative as a material tax risk?",


    # =========================================================================
    # BLOCK 30 — OUT-OF-SCOPE / NONE QUESTIONS [NONE]
    # Designed so the RAG system finds NO answers within any of the 10-K filings.
    # Useful for benchmark evaluation of hallucinations or out-of-context handling.
    # =========================================================================

    # Companies not included in the source documentation
    "What was NVIDIA's net income for its most recent fiscal year?",
    "How much revenue did Micron Technology (MU) generate from HBM sales during the fiscal year?",
    "What was Intel's revenue within its Foundry Services segment?",
    "How many electric vehicles did BYD sell globally in 2025?",
    "What was the operating margin for Samsung Semiconductor in 2025?",

    # Non-disclosed data in 10-Ks by nature (Forward-looking / Private details)
    "What will be Apple's stock price in December 2026?",
    "Which company in the group will achieve the highest total shareholder return in 2027?",
    "How much will Microsoft pay in dividends in the third quarter of 2027?",
    "What is the independent private market valuation of Google's DeepMind AI division?",
    "How much does Elon Musk earn personally from Tesla in gross monthly salary?",

    # Internal / Confidential Information
    "What is AWS's internal pricing or discount strategy for large enterprise cloud contracts?",
    "What are the specific terms and wafer pricing details in Apple's supply agreement with TSMC?",
    "How much capital did Meta pay out in discretionary performance bonuses to AI engineers in 2025?",
    "What is the actual net margin of Apple's App Store, excluding internal hosting costs?",
    "How many unannounced layoffs did Amazon execute in 2025 without public disclosure?",

    # Out-of-period data
    "How much net income did Microsoft report in the first quarter of 2019?",
    "What was Google's net profit in 2010 prior to its corporate restructuring into Alphabet?",
    "How many employees did Amazon have worldwide in 2015?",

    # Completely unrelated domains
    "What is the chemical formula for graphene oxide and what are its electrical properties?",
    "What was the final score of the Super Bowl match played in the year 2025?",
    "What are the daily sodium intake limitations recommended by the WHO?",
    "What was the total Gross Domestic Product (GDP) of Spain in 2025?",
    "What is the average orbital distance between Earth and Mars?",

    ]


    # =============================================================================
    # ESTADÍSTICAS RÁPIDAS
    # =============================================================================
    print(f"Total de queries: {len(query_tests)}")

    all_results = []

    for query in query_tests:
        print(f"\n\n=========== Running experiments for query: '{query}' ===========")
        pipeline_lc = RAGPipelineOrchestratorLC(texto_limpio) # Uses the 'texto_limpio' from previous cells
        pipeline_lc.ejecutar_matriz_completa(query=query)
        all_results.extend(pipeline_lc.resultados) # Collect results from each query
        # Removed pipeline_lc.mostrar_tabla_comparativa() from inside the loop
        pipeline_lc.mostrar_tabla_comparativa()

    # Display all responses for all queries at the end (cleaned up and using the 'query' field)
    print("\n\n=== DETALLE DE LAS RESPUESTAS Y CONTEXTO PARA TODAS LAS QUERIES (LangChain) ===")
    headers_responses = ["Query", "Config ID", "Chunking Strategy", "Embedding Model", "Retrieval Strategy", "ReRanker Model", "SLM Model", "Respuesta", "Longitud Respuesta (caracteres)", "Tokens Contexto Recuperados"]
    table_all_responses = []

    for r in all_results:
        table_all_responses.append([
            r.query, # Use the stored query directly
            r.config_id,
            r.chunk_strategy,
            r.embedding_model,
            r.retrieval_strategy,
            r.reranker_model,
            r.slm_model,
            r.respuesta,
            len(r.respuesta),
            r.tokens_recuperados
        ])

    # Sort results by Query, then by SLM model, then by chunking strategy for better readability
    table_all_responses_sorted = sorted(table_all_responses, key=lambda x: (x[0], x[6], x[2])) # x[0] is query, x[6] is SLM, x[2] is chunking
    print(tabulate(table_all_responses_sorted, headers=headers_responses, tablefmt="grid"))

    # Display overall statistics for execution times
    display_overall_statistics(all_results)


Total de queries: 3


=========== Running experiments for query: 'What are the company's primary business segments and products?' ===========
 Iniciando evaluación masiva local (LangChain): 72 permutaciones lógicas para la query: 'What are the company's primary business segments and products?'.
--------------------------------------------------------------------------------


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


=== MATRIZ DE COMPARATIVA DE RENDIMIENTO (ORDENADO POR VELOCIDAD LOCAL - LangChain) ===
+---------+----------------------+----------------------+-----------------+-------------------+------------------------+--------------+-------------------+----------------------------------------------------------------+
| ID      | Chunking             | Embedding            | Retrieval       | ReRanker          | SLM                    | Tiempo (s)   |   Tokens Contexto | Query                                                          |
+=========+======================+======================+=================+===================+========================+==============+===================+================================================================+
| EXP_004 | C2.1_RECURSIVE       | all-MiniLM-L6-v2     | R4.1_COSINE     | bge-reranker-base | Phi-3-mini-4k-instruct | 0.0265s      |               446 | What are the company's primary business segments and products? |
+---------+----------------

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

### Reinstalling `faiss-gpu-cu12` to resolve `ModuleNotFoundError`

Given the persistent `ModuleNotFoundError` for `faiss`, we will perform a thorough reinstallation to ensure a clean environment. This involves:

1.  **Force uninstalling** any `faiss` related packages to remove potential conflicts.
2.  **Clearing the pip cache** to eliminate any corrupted or old package data.
3.  **Reinstalling `faiss-gpu-cu12`**, which is suitable for your T4 GPU runtime.

After running the following cells, please re-run the main RAG pipeline cell (`E9Py2dufASKR`) to confirm the `faiss` module can be imported successfully.

In [ ]:
# Force uninstall both faiss-cpu and faiss-gpu-cu12 to ensure no conflicting versions remain
!pip uninstall faiss-cpu -y
!pip uninstall faiss-gpu-cu12 -y

In [ ]:
# Clear pip cache to remove any corrupted or residual package data
!pip cache purge

In [1]:
# Reinstall only faiss-gpu-cu12 for a clean installation
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 799.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 14.4 MB/s eta 0:00:00


In [2]:
# Verify faiss installation
import faiss
print("faiss imported successfully!")

faiss imported successfully!


After running the above cells, please re-run the main RAG pipeline cell (`E9Py2dufASKR`) to verify if the FAISS error is resolved.